# FiLM CNN-LSTM with Bilateral Insole Pressure and IMU
## Three-participant leave-one-subject-out evaluation

This notebook implements the dissertation's FiLM configuration with seed 42.
It estimates eight pelvis-relative, orientation-normalised 3D landmarks from
bilateral plantar pressure and six-axis foot IMUs. Video-derived SAM 3D Body
poses provide reference targets; video is not a model input at inference.

**Default architecture:** frame-wise pressure CNN → 256-D pressure embedding;
38-D bilateral IMU features → 64-D linear embedding → FiLM conditioning;
fused 256-D sequence → two-layer LSTM → scalar temporal attention → regression.

| Setting | Default implementation |
|---|---|
| Pressure input | 30 frames, two 66 × 30 pressure maps per frame |
| IMU input | Acceleration in g and angular velocity in degrees/s; 19 derived features per foot |
| IMU embedding | 38 → 64 dimensions; no separate temporal IMU encoder |
| Fusion | `(1 + gamma) * pressure_embedding + beta`; `USE_FILM=True` |
| IMU normalisation | `NORMALISE_IMU=False`; fixed feature-specific scaling |
| Temporal model | Two-layer unidirectional LSTM, hidden width 256, dropout 0.3 |
| Output | Final three poses in each window, eight landmarks × three coordinates; only the last pose is scored |
| Evaluation | Three LOSO folds with the three study participants; seed 42 |

Pressure-derived stance flags support ZUPT feature construction and the contact
regulariser. IMU measurements provide motion information that may complement
pressure during unloading and swing. The code does not enforce a particular
stance/swing interpretation of the learned FiLM scale and bias.

`USE_FILM=False` selects the concatenation control. It requires a separate
training run and is distinct from zeroing IMU input to a trained model.

The default input directory is `data/synced_levelled/`. Supply the three study
sessions with row-aligned `insoles`, `imu`, `joints`, `timestamps` and
`joint_names` arrays. Sessions without IMU are skipped. Prepare and verify the
upstream pressure correction, levelling and synchronisation separately.

The notebook contains MAE/MSE/MPJPE, a constant-pose baseline, amplitude and
correlation diagnostics, per-fold IMU ablation and FiLM statistics, plus optional
visualisation and angle diagnostics. Outputs are cleared; the dissertation's reported results
are documented in README, not stored here as results of a new run.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import random
import math
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Paths ─────────────────────────────────────────────────────────────────────
# BASE contains synced_levelled/ and the checkpoint output directory.
BASE       = Path("data")          # <-- change to your data root
SYNCED_DIR = BASE / "synced_levelled"
MODELS_DIR = BASE / "models_cnn_lstm_imu"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("Synced dir:", SYNCED_DIR)
print("Models dir:", MODELS_DIR)

## 2. Joint setup and skeleton connections

In [ ]:
# ── Which insole(s) feed the model, and which leg(s) it regresses ────────────
INPUT_FEET = ("left", "right")
# The dissertation uses both feet; alternative foot subsets require separate runs.

OUTPUT_SIDES = ("left", "right")
# The dissertation evaluates both sides. Some optional plots assume all eight landmarks.

FOOT_TO_CHANNEL = {"left": 0, "right": 1}
FOOT_IDX = [FOOT_TO_CHANNEL[f] for f in INPUT_FEET]
N_FEET = len(FOOT_IDX)

TARGET_JOINTS = [
    "left-hip",
    "right-hip",
    "left-knee",
    "right-knee",
    "left-heel",
    "right-heel",
    "left-big-toe-tip",
    "right-big-toe-tip",
]

CONNECTIONS = [
    ("left-hip", "left-knee"),
    ("left-knee", "left-heel"),
    ("left-heel", "left-big-toe-tip"),
    ("right-hip", "right-knee"),
    ("right-knee", "right-heel"),
    ("right-heel", "right-big-toe-tip"),
    ("left-hip", "right-hip"),
]

def as_str_list(names):
    return [n.decode("utf-8") if isinstance(n, bytes) else str(n) for n in names]

# Keep the canonical order, then drop the sides we are not regressing.
OUTPUT_JOINTS = [j for j in TARGET_JOINTS if j.split("-")[0] in OUTPUT_SIDES]
if not OUTPUT_JOINTS:
    raise ValueError(f"OUTPUT_SIDES={OUTPUT_SIDES} selected no joints from {TARGET_JOINTS}")

CONNECTIONS = [(a, b) for a, b in CONNECTIONS
               if a in OUTPUT_JOINTS and b in OUTPUT_JOINTS]

print(f"Input feet   : {INPUT_FEET}  -> {N_FEET} insole channel(s)")
print(f"Output joints: {len(OUTPUT_JOINTS)}  {OUTPUT_JOINTS}")

# A short tag for the current input configuration, appended to every artefact
# path so "both" / "left" / "right" runs sit side by side instead of clobbering
# one another.
FOOT_TAG = "".join(f[0] for f in INPUT_FEET).upper()          # LR / L / R
SIDE_TAG = "".join(s[0] for s in OUTPUT_SIDES).upper()
RUN_TAG  = f"in{FOOT_TAG}_out{SIDE_TAG}"
print(f"Run tag      : {RUN_TAG}")

# ── Which IMU representation the model is fed ──────────────────────────
# This sits with the other input decisions rather than in the model config,
# because section 4 builds the features and runs before that config exists.
USE_ZUPT_FEATURES = True
# True  -> the 19-D feature block in section 4: acceleration direction,
#          ZUPT-corrected velocity, stride-relative displacement, gait phase.
# False -> the raw 6 axes, as before.

IMU_RAW_AXES  = 6
IMU_ZUPT_DIMS = 19
print(f"IMU input    : {'ZUPT 19-D' if USE_ZUPT_FEATURES else 'raw 6-axis'}")


# ── How the IMU is scaled before it reaches the model ──────────────────
# Defined here, not in the model-config cell, for the same reason
# USE_ZUPT_FEATURES is: section 4 builds the features and runs first.
IMU_SCALE_MODE = "scale"
# Raw-axis scaling follows BALANCE's MBALANCE/code/imu_norm.py.
# In 'scale' mode acceleration stays in g and gyro is divided by GYRO_SCALE.
# In 'none' mode the six input axes are retained.
# This option is bypassed when USE_ZUPT_FEATURES=True: the feature extractor
# applies its own fixed divisors. No IMU mean/std is fitted in the default run.

GYRO_SCALE = 200.0        # deg/s, the divisor IMUFeatureExtractor already uses

if IMU_SCALE_MODE not in ("scale", "none"):
    raise ValueError(f"IMU_SCALE_MODE must be 'scale' or 'none', got {IMU_SCALE_MODE!r}")

print(f"IMU scaling  : {IMU_SCALE_MODE}"
      + ("  (no-op on the ZUPT path - see section 4.1)" if USE_ZUPT_FEATURES
         else f"  (active: accel in g, gyro / {GYRO_SCALE:.0f})"))

## 3. Load synced sessions

In [ ]:
def load_synced_sessions(synced_dir: Path):
    """Load every synced session, reporting which ones carry an IMU stream."""
    session_files = sorted(synced_dir.glob("*_synced.npz"))
    if not session_files:
        raise FileNotFoundError(f"No synced files found in {synced_dir}")

    sessions = []
    for path in session_files:
        data = np.load(path, allow_pickle=True)
        insoles     = np.asarray(data["insoles"],     dtype=np.float32)
        joints      = np.asarray(data["joints"],      dtype=np.float32)
        joint_names = as_str_list(data["joint_names"])
        timestamps  = np.asarray(data["timestamps"],  dtype=np.float64)

        # (T, 2, 6) = 2 feet x (ax, ay, az, gx, gy, gz); absent on older sessions.
        imu = np.asarray(data["imu"], dtype=np.float32) if "imu" in data.files else None

        sessions.append({
            "name":        path.stem.replace("_synced", ""),
            "insoles":     insoles,
            "joints":      joints,
            "joint_names": joint_names,
            "timestamps":  timestamps,
            "imu":         imu,
        })
        tag = f"imu={imu.shape}" if imu is not None else "imu=MISSING"
        print(f"{path.name}: insoles={insoles.shape}, joints={joints.shape}, {tag}")
    return sessions

sessions = load_synced_sessions(SYNCED_DIR)

with_imu = [s["name"] for s in sessions if s["imu"] is not None]
without  = [s["name"] for s in sessions if s["imu"] is None]
print(f"\nSessions with IMU    ({len(with_imu)}): {with_imu}")
print(f"Sessions without IMU ({len(without)}): {without}")

if not with_imu:
    raise RuntimeError(
        "No session carries an 'imu' array, so there is nothing to fuse.\n"
        "Produce one per session with:\n"
        "    python insole_collection/offline_processing_imu.py <session>\n"
        "stage the IMU with the pressure stream on a shared clock, then rebuild "
        "the configured synchronised dataset. See the README data workflow."
    )
assert len(with_imu) >= 2, (
    f"Need at least 2 IMU-bearing subjects for LOSO, found {len(with_imu)}."
)


## 4. Session preprocessing

Non-finite pressure and IMU values are replaced by zero. Pressure is not
z-scored. Each foot's six-axis IMU is expanded to 19 features: acceleration,
scaled angular velocity, normalised acceleration direction, angular-speed
magnitude, smoothed forward acceleration, ZUPT velocity, scaled within-swing
displacement, and sine/cosine gait phase. The extractor uses pressure-derived
stance and a fixed time step estimated from the session's median timestamp gap.
Targets are selected by landmark name; their fold-specific normalisation is
applied later in the LOSO loop.

In [ ]:
def replace_nan_with_zero(x):
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def build_joint_index(joint_names):
    mapping = {name: i for i, name in enumerate(joint_names)}
    missing = [name for name in TARGET_JOINTS if name not in mapping]
    if missing:
        raise ValueError(f"Missing required joints: {missing}")
    return mapping

def select_output_targets(joints, joint_names):
    name_to_idx  = build_joint_index(joint_names)
    keep_indices = [name_to_idx[name] for name in OUTPUT_JOINTS]
    return joints[:, keep_indices, :].astype(np.float32)

# ═════════════════════════════════════════════════════════════════════════
# Stance detection and the ZUPT feature block
# ═════════════════════════════════════════════════════════════════════════
from scipy.signal import savgol_filter


def estimate_fps(timestamps, default=15.0):
    """Median sample rate, tolerating timestamps in either seconds or ms."""
    t = np.asarray(timestamps, dtype=np.float64).ravel()
    if t.size < 3:
        return default
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size == 0:
        return default
    med = float(np.median(dt))
    if med > 1.0:            # milliseconds
        med /= 1000.0
    if not (1e-4 < med < 1.0):
        return default
    return 1.0 / med


def stance_from_pressure(x, frac=0.25, min_run=2):
    """Per-foot pressure-derived contact mask, shape (T, n_feet).
    Use q10 + frac * (q90 - q10) of each foot's session load. Interior
    runs shorter than min_run are replaced by the preceding state.
    Near-constant load is assigned stance=True by this implementation."""
    T, n_feet = x.shape[0], x.shape[1]
    total = x.reshape(T, n_feet, -1).sum(axis=-1)          # (T, n_feet)
    stance = np.zeros((T, n_feet), dtype=bool)

    for f in range(n_feet):
        lo = np.percentile(total[:, f], 10)
        hi = np.percentile(total[:, f], 90)
        if hi - lo < 1e-9:                                  # near-constant load; use the configured stance fallback
            stance[:, f] = True
            continue
        stance[:, f] = total[:, f] > (lo + frac * (hi - lo))

        # de-flicker
        s = stance[:, f]
        start = 0
        for t in range(1, T + 1):
            if t == T or s[t] != s[start]:
                if t - start < min_run and start > 0 and t < T:
                    s[start:t] = s[start - 1]
                start = t
        stance[:, f] = s

    return stance


class IMUFeatureExtractor:
    """Raw (T, n_feet, 6) [g, deg/s] -> (T, n_feet, 19) features.
    
    [0:3] acceleration in g; [3:6] gyro/200; [6:9] unit acceleration
    direction (a gravity-direction proxy); [9] gyro magnitude/100;
    [10] smoothed sensor-x acceleration/2; [11:14] ZUPT velocity;
    [14:17] within-swing displacement/stride_scale; [17:19] phase sin/cos.
    
    Integration subtracts a fixed 1 g sensor-z bias and uses a constant dt.
    Each detected swing has its integration baseline and linear drift
    removed; stance velocity and displacement are set to zero. These are
    sensor-derived features, not a reconstructed world-frame trajectory."""

    def __init__(self, fps=15.0, smooth_window=5, stride_scale=0.2):
        self.fps = fps
        self.dt = 1.0 / fps
        self.smooth_window = smooth_window
        self.stride_scale = stride_scale
        self.g = 9.81

    def __call__(self, imu, stance_mask):
        T, n_feet, _ = imu.shape
        out = np.zeros((T, n_feet, 19), dtype=np.float32)

        win = min(self.smooth_window, T if T % 2 else T - 1)
        if win is not None and win < 3:
            win = 0

        for f in range(n_feet):
            acc = imu[:, f, 0:3].astype(np.float32)
            gyr = imu[:, f, 3:6].astype(np.float32)
            stance = stance_mask[:, f].astype(bool)

            acc_smooth = savgol_filter(acc, win, 2, axis=0) if win else acc

            gravity = acc / (np.linalg.norm(acc, axis=1, keepdims=True) + 1e-6)
            gyr_mag = np.linalg.norm(gyr, axis=1) / 100.0
            fwd = acc_smooth[:, 0] / 2.0

            vel, disp = self._zupt(acc_smooth, stance)
            phase = self._gait_phase(stance)

            out[:, f, :] = np.concatenate([
                acc,                    # [0:3]
                gyr / 200.0,            # [3:6]
                gravity,                # [6:9]
                gyr_mag[:, None],       # [9]
                fwd[:, None],           # [10]
                vel,                    # [11:14]
                disp,                   # [14:17]
                phase,                  # [17:19]
            ], axis=1)

        return out

    def _zupt(self, acc, stance):
        T = len(acc)
        a = acc.copy()
        a[:, 2] -= 1.0                                   # subtract the fixed sensor-z gravity approximation
        vel = np.cumsum(a * self.g * self.dt, axis=0).astype(np.float32)

        # Remove each detected swing's preceding integration baseline and
        # terminal drift; stance samples are zeroed after all swing corrections.
        for s0, s1 in self._swings(stance):
            span = max(s1 - s0 + 1, 1)
            base = vel[s0 - 1].copy() if s0 > 0 else np.zeros(3, dtype=np.float32)
            drift = vel[s1] - base
            for t in range(s0, s1 + 1):
                vel[t] = vel[t] - base - drift * ((t - s0 + 1) / span)

        vel[stance] = 0.0

        disp = np.cumsum(vel * self.dt, axis=0).astype(np.float32)
        for s0, s1 in self._swings(stance):
            if s0 > 0:
                disp[s0:s1 + 1] -= disp[s0 - 1]
        disp[stance] = 0.0

        return vel, (disp / self.stride_scale).astype(np.float32)

    @staticmethod
    def _swings(stance):
        """[(first, last)] index pairs for each contiguous swing."""
        segs, in_swing, s0 = [], False, 0
        for t, on_ground in enumerate(stance):
            if not on_ground and not in_swing:
                in_swing, s0 = True, t
            elif on_ground and in_swing:
                segs.append((s0, t - 1))
                in_swing = False
        if in_swing:
            segs.append((s0, len(stance) - 1))
        return segs

    @staticmethod
    def _gait_phase(stance):
        """Phase in [0, 2pi) measured between successive heel strikes."""
        T = len(stance)
        phase = np.zeros(T, dtype=np.float32)
        strikes = np.where(np.diff(stance.astype(np.int8)) == 1)[0] + 1

        if len(strikes) < 2:
            phase[:] = 2 * np.pi * np.arange(T) / max(T, 1)
        else:
            strides = np.diff(strikes)
            for k, t0 in enumerate(strikes[:-1]):
                span = max(int(strides[k]), 1)
                for t in range(t0, min(strikes[k + 1], T)):
                    phase[t] = 2 * np.pi * (t - t0) / span
            span = int(np.median(strides))
            for t in range(strikes[-1], T):
                phase[t] = 2 * np.pi * ((t - strikes[-1]) % span) / span
            span0 = int(strides[0])
            for t in range(0, strikes[0]):
                phase[t] = 2 * np.pi * ((t - strikes[0]) % span0) / span0

        return np.stack([np.sin(phase), np.cos(phase)], axis=1).astype(np.float32)


def apply_imu_scale(imu_raw, mode=None, gyro_scale=None):
    """Fixed scaling for the optional raw six-axis path, following BALANCE
    imu_norm.py. Acceleration remains in g; gyro is divided by gyro_scale
    in scale mode. No fitted statistics are used. The default ZUPT path
    bypasses this helper and applies scaling inside IMUFeatureExtractor."""
    mode = IMU_SCALE_MODE if mode is None else mode
    gyro_scale = GYRO_SCALE if gyro_scale is None else gyro_scale
    if mode == "none":
        return imu_raw.astype(np.float32, copy=True)
    out = imu_raw.astype(np.float32, copy=True)
    out[:, :, 3:6] /= gyro_scale          # accel [0:3] left alone, in g
    return out


def preprocess_sessions(sessions):
    """Keep only IMU-bearing sessions and stage X / IMU / Y for the LOSO loop.

    "X"          insole pressure maps,  (T, n_feet, 66, 30) float32
    "IMU"        model input,           (T, n_feet, 19) ZUPT block, or (T, n_feet, 6) raw
    "IMU_raw"    untouched 6-axis copy, used for the sanity check below
    "IMU_STANCE" contact mask,          (T, n_feet) bool, drives ZUPT and the contact loss
    "Y"          joint 3D coords,       (T, J, 3) float32 -> per-fold z-scored later
    "Y_raw"      untouched copy of Y, used for reporting errors in real units
    """
    processed = []
    for session in sessions:
        if session["imu"] is None:
            print(f"{session['name']}: no IMU - skipped")
            continue

        x = replace_nan_with_zero(session["insoles"])[:, FOOT_IDX]   # keep INPUT_FEET only
        imu = replace_nan_with_zero(session["imu"])[:, FOOT_IDX]     # same feet as the pressure
        y = select_output_targets(session["joints"], session["joint_names"])

        if len(imu) != len(x):
            raise ValueError(
                f"{session['name']}: IMU has {len(imu)} rows but pressure has {len(x)}. "
                "sync_session() writes them row-aligned, so this file is stale - "
                "re-run part2_preprocessing.ipynb."
            )

        stance = stance_from_pressure(x)

        if USE_ZUPT_FEATURES:
            fps = estimate_fps(session["timestamps"])
            imu_feat = IMUFeatureExtractor(fps=fps)(imu, stance)
        else:
            fps = estimate_fps(session["timestamps"])
            imu_feat = apply_imu_scale(imu)

        processed.append({
            "name":       session["name"],
            "X":          x,
            "IMU":        imu_feat,
            "IMU_raw":    imu.copy(),
            "IMU_STANCE": stance,
            "Y":          y,
            "Y_raw":      y.copy(),
            "timestamps": session["timestamps"],
        })
        print(f"{session['name']}: X={x.shape}, IMU={imu_feat.shape}, Y={y.shape}  "
              f"[{fps:.1f} Hz, stance {stance.mean():.0%}]")
    return processed

processed_sessions = preprocess_sessions(sessions)

# Inspect raw acceleration and gyro magnitudes before training.
# Constant readings can reflect a stationary interval or sensor problems;
# these descriptive statistics do not determine sensor validity by themselves.
print("\nIMU sanity check (accel in g, gyro in deg/s):")
for s in processed_sessions:
    a = np.linalg.norm(s["IMU_raw"][:, :, 0:3], axis=2)
    g = np.linalg.norm(s["IMU_raw"][:, :, 3:6], axis=2)
    parts = [f"{foot}: |a|={np.median(a[:, k]):.2f} g, |g| p95={np.percentile(g[:, k], 95):.0f} deg/s"
             for k, foot in enumerate(INPUT_FEET)]
    print(f"  {s['name']:<12} " + "   ".join(parts))

# Inspect the derived velocity spread. Stance velocity is explicitly set
# to zero in _zupt; this check alone does not validate contact detection.
if USE_ZUPT_FEATURES:
    print("\nZUPT check (velocity spread, m/s):")
    for s in processed_sessions:
        v = s["IMU"][:, :, 11:14]
        st = s["IMU_STANCE"]
        v_st = v[st].std() if st.any() else float("nan")
        v_sw = v[~st].std() if (~st).any() else float("nan")
        flag = "ok" if v_st < 0.5 * v_sw else "SUSPECT - check the stance threshold"
        print(f"  {s['name']:<12} stance={v_st:.4f}  swing={v_sw:.4f}   {flag}")

## 4.1 IMU feature-scale diagnostic

Print the standard deviation of each model-input IMU feature and the pressure
input. The `ratio` column compares each IMU feature with the median nonzero IMU
feature standard deviation. The 0.1–10 range is a diagnostic threshold, not a
test of predictive value or a guarantee about optimisation. On the default
19-feature path, scaling is performed inside `IMUFeatureExtractor`;
`IMU_SCALE_MODE` affects only the optional raw six-axis path.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# IMU scale audit — measure the balance instead of assuming it
# ═════════════════════════════════════════════════════════════════════════
"""Descriptive feature-scale check. Ratios are relative to the median
nonzero IMU feature standard deviation. Flags identify constant or
differently scaled features for inspection; they do not prove a sensor
failure, implementation bug, or a particular gradient contribution."""

_ZUPT_DIM_NAMES = (
    ["accel_x", "accel_y", "accel_z"]
    + ["gyro_x/200", "gyro_y/200", "gyro_z/200"]
    + ["grav_x", "grav_y", "grav_z"]
    + ["|gyro|/100", "fwd_accel"]
    + ["vel_x", "vel_y", "vel_z"]
    + ["disp_x", "disp_y", "disp_z"]
    + ["phase_sin", "phase_cos"]
)
_RAW_DIM_NAMES = ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z"]

dim_names = _ZUPT_DIM_NAMES if USE_ZUPT_FEATURES else _RAW_DIM_NAMES
all_imu = np.concatenate([s["IMU"] for s in processed_sessions], axis=0)   # (T, n_feet, D)
all_x   = np.concatenate([s["X"]   for s in processed_sessions], axis=0)

stds = all_imu.reshape(-1, all_imu.shape[-1]).std(axis=0)
med  = float(np.median(stds[stds > 0])) if np.any(stds > 0) else 1.0

print(f"Pressure input std : {all_x.std():.4f}   (range "
      f"[{all_x.min():.3f}, {all_x.max():.3f}])")
print(f"IMU representation : {'ZUPT 19-D' if USE_ZUPT_FEATURES else 'raw 6-axis'}, "
      f"IMU_SCALE_MODE={IMU_SCALE_MODE}")
print(f"IMU median dim std : {med:.4f}\n")
print(f"  {'dim':<4} {'name':<14} {'std':>10} {'ratio':>8}   flag")
print("  " + "-" * 48)
for i, (name, sd) in enumerate(zip(dim_names, stds)):
    ratio = sd / med if med > 0 else float("nan")
    if sd == 0:
        flag = "CONSTANT - inspect signal"
    elif ratio > 10 or ratio < 0.1:
        flag = "OUT OF BAND - inspect scale"
    else:
        flag = "ok"
    print(f"  [{i:>2}] {name:<14} {sd:>10.4f} {ratio:>7.2f}x   {flag}")

_bad = [(i, dim_names[i]) for i, sd in enumerate(stds)
        if sd == 0 or not (0.1 <= sd / med <= 10)]
print()
if _bad:
    print(f"{len(_bad)} dimension(s) out of band: "
          + ", ".join(f"[{i}] {n}" for i, n in _bad))
    print("Inspect the input units and feature distributions. These flags do not "
          "by themselves establish an implementation error.")
else:
    print("All IMU feature standard deviations are between 0.1 and 10 times "
          "the median nonzero IMU feature standard deviation.")

del all_imu, all_x

## 5. Model configuration

The dissertation configuration uses 30-frame windows, batch size 32, seed 42,
FiLM fusion and three output poses. IMU z-scoring is disabled. The loss is
`L_kinematic + 0.5 L_contact + 0.3 L_bone + 0.1 L_smooth`.
Training uses Adam, weight decay `1e-5`, at most 700 epochs and early-stopping
patience 400. CNN/LSTM/other learning rates are `2e-3`/`3e-3`/`1e-3`.
The training loop uses zero DataLoader workers and clips gradient norm to 1.0.

In [ ]:
SEQ_LEN          = 30
SEQ_STRIDE       = 1
BATCH_SIZE       = 32
NUM_WORKERS      = 2
CNN_CHANNELS     = (16, 32)
LSTM_HIDDEN_SIZE = 256
LSTM_LAYERS      = 2
DROPOUT          = 0.3
LR               = 1e-3
WEIGHT_DECAY     = 1e-5
EPOCHS           = 700
PATIENCE         = 400
VAL_RATIO        = 0.15

# ── IMU ────────────────────────────────────────────────────
PRESSURE_EMBED = 256      # width of the pressure branch before fusion

IMU_EMBED_DIM = 64
# Width of the linear IMU embedding used to generate FiLM scale and bias.

# USE_ZUPT_FEATURES / IMU_RAW_AXES / IMU_ZUPT_DIMS are set in section 2, next to
# the other input choices, because section 4 needs them before this cell runs.

NORMALISE_IMU = False
# The dissertation configuration leaves the fixed-scaled IMU features
# unstandardised. Optional raw-axis runs may enable fold-specific z-scoring.
# The guard below prevents enabling it with USE_ZUPT_FEATURES=True.

USE_FILM = True
# True  -> IMU conditions the pressure embedding: y = (1 + gamma) * p + beta
# False -> concatenate the branches, i.e. the previous behaviour, kept as the
#          ablation control.

# ── Physics-constrained loss ─────────────────────────────────────
PRED_FRAMES = 3
# The head regresses the last PRED_FRAMES poses of the window rather than the
# final one alone. Frame -1 is the pose that gets scored; the extra frames exist
# so the contact and smoothness terms have a real predicted trajectory to
# differentiate. At PRED_FRAMES = 1 both terms are identically zero.

AUX_FRAME_WEIGHT = 0.5   # kinematic weight on the auxiliary frames vs frame -1

USE_PHYSICS_LOSS = True
PHYSICS_WEIGHTS  = {
    "kinematic": 1.0,    # weighted L1 against the marker positions
    "contact":   0.5,    # squared heel/toe displacement during detected stance
    "bone":      0.3,    # segment lengths hold at the training-fold median
    "smooth":    0.1,    # mean absolute second temporal difference
}

# Derived from INPUT_FEET (see the joint-setup cell).
PRESSURE_CHANNELS = N_FEET   # one channel per foot
IMU_PER_FOOT_DIM  = IMU_ZUPT_DIMS if USE_ZUPT_FEATURES else IMU_RAW_AXES
IMU_FEAT_DIM      = IMU_PER_FOOT_DIM * N_FEET

if USE_ZUPT_FEATURES and NORMALISE_IMU:
    raise ValueError(
        "This configuration does not combine IMU z-scoring with ZUPT features. "
        "Set one of the two to False."
    )

print("Config OK")
print(f"  SEQ_LEN={SEQ_LEN}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, PATIENCE={PATIENCE}")
print(f"  CNN_CHANNELS={CNN_CHANNELS}, LSTM_HIDDEN_SIZE={LSTM_HIDDEN_SIZE}, LSTM_LAYERS={LSTM_LAYERS}")
print(f"  IMU: {'ZUPT 19-D' if USE_ZUPT_FEATURES else 'raw 6-axis'} x {N_FEET} feet"
      f" -> IMU_FEAT_DIM={IMU_FEAT_DIM} -> IMU_EMBED_DIM={IMU_EMBED_DIM}")
print(f"  IMU_SCALE_MODE={IMU_SCALE_MODE}"
      + ("  (no-op on the ZUPT path - see section 4.1)" if USE_ZUPT_FEATURES
         else "  (active: accel in g, gyro / 200)"))
print(f"  NORMALISE_IMU={NORMALISE_IMU}")
print(f"  Fusion: {'FiLM (IMU modulates pressure)' if USE_FILM else 'concat (control)'}")
print(f"  Physics loss: {USE_PHYSICS_LOSS}, PRED_FRAMES={PRED_FRAMES}, weights={PHYSICS_WEIGHTS}")
print(f"  INPUT_FEET={INPUT_FEET} -> PRESSURE_CHANNELS={PRESSURE_CHANNELS}")

## 6. Build temporal sequences

In [ ]:
class SlidingWindowDataset(Dataset):
    """Windows of pressure + IMU ending at the frame whose pose we regress."""

    def __init__(self, sessions, items, seq_len):
        self.sessions = sessions
        self.items    = items
        self.seq_len  = seq_len

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        session_idx, end_idx = self.items[idx]
        s         = self.sessions[session_idx]
        start_idx = end_idx - self.seq_len + 1

        x = s["X"][start_idx : end_idx + 1]                       # (T, n_feet, H, W)
        x_input = x

        # IMU over the same window, kept per-foot as (T, n_feet, D) so the
        # left/right split stays visible; the model flattens it to (T, n_feet*D).
        imu = s["IMU"][start_idx : end_idx + 1]

        # The physics terms need a short predicted trajectory, not one pose, so
        # the last PRED_FRAMES targets and their contact flags come along. The
        # scored frame is still end_idx, i.e. index -1.
        first = end_idx - PRED_FRAMES + 1
        y      = s["Y"][first : end_idx + 1]                      # (K, J, 3)
        stance = s["IMU_STANCE"][first : end_idx + 1]             # (K, n_feet)
        y_raw  = s["Y_raw"][end_idx]

        return {
            "X":       torch.from_numpy(x_input).float(),
            "IMU":     torch.from_numpy(imu).float(),
            "Y":       torch.from_numpy(y).float(),
            "STANCE":  torch.from_numpy(stance).bool(),
            "Y_raw":   torch.from_numpy(y_raw).float(),
            "session": s["name"],
        }


def build_items(sessions_subset, seq_len):
    """(session_idx, end_idx) pairs, one window per frame."""
    items = []
    for local_i, s in enumerate(sessions_subset):
        T = len(s["X"])
        if T < max(seq_len, PRED_FRAMES):
            print(f"  WARNING: {s['name']} has only {T} frames < SEQ_LEN={seq_len}, skipped.")
            continue
        for end_idx in range(seq_len - 1, T):
            items.append((local_i, end_idx))
    return items

print("Dataset helpers defined.")
print(f"Pressure channels per frame: {PRESSURE_CHANNELS}")
print(f"IMU features per frame     : {IMU_FEAT_DIM}  "
      f"({N_FEET} feet x {IMU_PER_FOOT_DIM} {'ZUPT' if USE_ZUPT_FEATURES else 'raw'} dims)")
print(f"Targets per window         : {PRED_FRAMES} frame(s), scored on the last")


## 7. FiLM model and training objective

The shared frame CNN maps two pressure channels through widths 16 and 32,
global average pooling and a linear projection to 256 dimensions. The bilateral
38-D IMU block is linearly embedded into 64 dimensions. A FiLM generator
(64 → 128 → 512, with batch normalisation, dropout and ReLU) produces 256-D
scale and bias vectors. The fused sequence enters a two-layer, 256-wide LSTM;
scalar softmax temporal attention pools its states before the regression head.
The FiLM output layer uses the default initialisation, so identity modulation
at initialisation is not guaranteed.

The kinematic term is weighted L1 over three poses, with frame weights
`(0.5, 0.5, 1.0)` and hip/knee/heel-and-toe weights `0.5/1/2`.
The contact term penalises squared heel/toe displacement during detected stance
in normalised target coordinates. The bone term uses inverse-normalised lengths,
seven connections, development-participant median references and relative
Smooth-L1 (`beta=0.05`). Smoothness is the mean absolute second temporal
difference. These are the implemented regularisers; the targets remain
pelvis-relative rather than world-coordinate trajectories.

FiLM reference: Perez et al., *FiLM: Visual Reasoning with a General
Conditioning Layer*, AAAI 2018.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# Physics-constrained loss
# ═════════════════════════════════════════════════════════════════════════
"""Composite objective used for both training and validation.
The head emits three poses; only the last is scored by position metrics.
Contact and smoothness operate on normalised, pelvis-relative predictions.
Bone regularisation uses inverse-normalised lengths relative to median
reference lengths from the two non-test participants. These terms add
no inference-time parameters."""

import torch
import torch.nn as nn
import torch.nn.functional as F


def _contact_index():
    """[(foot_channel, heel_joint, toe_joint)] for feet we both sense and regress."""
    out = []
    for f, side in enumerate(INPUT_FEET):
        heel, toe = f"{side}-heel", f"{side}-big-toe-tip"
        hi = OUTPUT_JOINTS.index(heel) if heel in OUTPUT_JOINTS else None
        ti = OUTPUT_JOINTS.index(toe) if toe in OUTPUT_JOINTS else None
        if hi is None and ti is None:
            continue
        out.append((f, hi, ti))
    return out


CONTACT_INDEX  = _contact_index()
BONE_PAIRS_IDX = [(OUTPUT_JOINTS.index(a), OUTPUT_JOINTS.index(b)) for a, b in CONNECTIONS]

print(f"Contact-constrained feet : {[(INPUT_FEET[f], h, t) for f, h, t in CONTACT_INDEX]}")
print(f"Bone pairs               : {[(OUTPUT_JOINTS[a], OUTPUT_JOINTS[b]) for a, b in BONE_PAIRS_IDX]}")


class PhysicalLosses:
    """Constraint terms. Every input is (B, K, J, 3) - batch, frames, joints, xyz."""

    @staticmethod
    def contact_loss(pred_seq, stance_seq):
        """Penalise squared heel/toe displacement during pressure-derived stance.
        The mask uses the later frame of each consecutive pair. Coordinates
        are normalised and pelvis-relative; this is the implemented contact
        regulariser, not an explicit world-frame no-slip constraint."""
        B, K = pred_seq.shape[0], pred_seq.shape[1]
        if K < 2 or not CONTACT_INDEX:
            return pred_seq.new_zeros(())

        disp = pred_seq[:, 1:] - pred_seq[:, :-1]            # (B, K-1, J, 3)
        num = pred_seq.new_zeros(())
        den = pred_seq.new_zeros(())

        for foot, heel, toe in CONTACT_INDEX:
            motion = pred_seq.new_zeros(disp.shape[0], disp.shape[1])
            if heel is not None:
                motion = motion + disp[:, :, heel].pow(2).sum(-1)
            if toe is not None:
                motion = motion + disp[:, :, toe].pow(2).sum(-1)
            st = stance_seq[:, 1:, foot].to(pred_seq.dtype)   # (B, K-1)
            num = num + (st * motion).sum()
            den = den + st.sum()

        return num / den.clamp(min=1.0)

    @staticmethod
    def bone_length_loss(pred_seq_real, bone_ref):
        """Segment lengths hold at the training fold's median.

        Scored as relative deviation, so the term is unit-free and a femur and a
        toe segment contribute comparably instead of the long bones dominating.
        """
        if not BONE_PAIRS_IDX:
            return pred_seq_real.new_zeros(())

        loss = pred_seq_real.new_zeros(())
        for k, (a, b) in enumerate(BONE_PAIRS_IDX):
            length = (pred_seq_real[:, :, a] - pred_seq_real[:, :, b]).norm(dim=-1)
            rel = (length - bone_ref[k]) / bone_ref[k].clamp(min=1e-6)
            loss = loss + F.smooth_l1_loss(rel, torch.zeros_like(rel), beta=0.05)
        return loss / len(BONE_PAIRS_IDX)

    @staticmethod
    def smoothness_loss(pred_seq):
        """Mean absolute second temporal difference over the predicted poses."""
        if pred_seq.shape[1] < 3:
            return pred_seq.new_zeros(())
        vel = pred_seq[:, 1:] - pred_seq[:, :-1]
        acc = vel[:, 1:] - vel[:, :-1]
        return acc.abs().mean()


def weighted_l1_seq(pred_seq, target_seq):
    """Weighted coordinate L1 over predicted poses. The final pose has weight
    1 and earlier poses AUX_FRAME_WEIGHT. Joint weights follow OUTPUT_JOINTS."""
    K = pred_seq.shape[1]
    w = JOINT_WEIGHTS.to(pred_seq.device)                       # (J,)
    per = (pred_seq - target_seq).abs().mean(-1)                # (B, K, J)
    per = (per * w).mean(-1)                                    # (B, K)

    fw = torch.full((K,), AUX_FRAME_WEIGHT, device=pred_seq.device, dtype=per.dtype)
    fw[-1] = 1.0
    return (per * fw).sum(dim=1).mean() / fw.sum()


def combined_physical_loss(pred_seq, target_seq, stance_seq, phys, weights=None):
    """Marker loss plus the three constraints.

    pred_seq / target_seq : (B, K, J, 3), normalised
    stance_seq            : (B, K, n_feet) bool
    phys                  : dict from make_physics_context() - y_mean, y_std, bone_ref
    """
    weights = weights or PHYSICS_WEIGHTS

    L_kin = weighted_l1_seq(pred_seq, target_seq)

    if weights.get("contact", 0.0) and stance_seq is not None:
        L_contact = PhysicalLosses.contact_loss(pred_seq, stance_seq)
    else:
        L_contact = pred_seq.new_zeros(())

    if weights.get("bone", 0.0) and phys is not None:
        pred_real = pred_seq * phys["y_std"] + phys["y_mean"]
        L_bone = PhysicalLosses.bone_length_loss(pred_real, phys["bone_ref"])
    else:
        L_bone = pred_seq.new_zeros(())

    if weights.get("smooth", 0.0):
        L_smooth = PhysicalLosses.smoothness_loss(pred_seq)
    else:
        L_smooth = pred_seq.new_zeros(())

    parts = {
        "kinematic": L_kin,
        "contact":   L_contact,
        "bone":      L_bone,
        "smooth":    L_smooth,
    }
    total = sum(weights.get(k, 0.0) * v for k, v in parts.items())
    return total, parts


def make_physics_context(y_mean, y_std, train_Y_raw):
    """Per-fold constants for the physics terms.

    The bone references are medians over the *training* subjects only, so the
    held-out subject's build never leaks into the constraint. The median rather
    than the mean because a handful of dropped-marker frames would otherwise
    stretch the reference.
    """
    ref = []
    for a, b in BONE_PAIRS_IDX:
        d = np.linalg.norm(train_Y_raw[:, a] - train_Y_raw[:, b], axis=-1)
        ref.append(float(np.median(d)))

    return {
        "y_mean":   torch.as_tensor(y_mean, dtype=torch.float32, device=DEVICE),
        "y_std":    torch.as_tensor(y_std,  dtype=torch.float32, device=DEVICE),
        "bone_ref": torch.as_tensor(ref,    dtype=torch.float32, device=DEVICE),
        "bone_ref_list": ref,
    }


# ════════════════════════════════════════════════════════════════════════════════════
# FiLM conditioning of frame-wise pressure embeddings
# ════════════════════════════════════════════════════════════════════════════════════
"""Feature-wise linear modulation: (1 + gamma) * pressure + beta.
The IMU embedding generates both vectors; no phase-specific sign pattern is
imposed. Reference: Perez et al., AAAI 2018, "FiLM: Visual Reasoning with a
General Conditioning Layer".
"""

class FiLMFusion(nn.Module):
    """
    Feature-wise Linear Modulation 融合层

    将 IMU 特征变换为调制系数（gamma, beta），用于动态调节压力特征
    """

    def __init__(self, imu_dim, pressure_dim, hidden_dim=128, dropout=0.2):
        """imu_dim is the IMU embedding width (64 in the default configuration);
        pressure_dim is 256 and hidden_dim is 128."""
        super().__init__()

        self.pressure_dim = pressure_dim

        # MLP: IMU → [gamma, beta]
        # 结构: imu_dim → hidden_dim (ReLU) → 2*pressure_dim
        self.mlp = nn.Sequential(
            nn.Linear(imu_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2 * pressure_dim),
        )

    def forward(self, x_pressure, x_imu):
        """Condition pressure embeddings (B,T,pressure_dim) using IMU embeddings
        (B,T,imu_dim). Flatten batch/time for the shared generator, split its
        output into gamma/beta, then apply (1+gamma)*pressure+beta."""
        B, T, D_i = x_imu.shape
        D_p = self.pressure_dim

        # 将 (B, T, dim) 变成 (B*T, dim) 通过 MLP
        x_imu_flat = x_imu.view(-1, D_i)
        gamma_beta = self.mlp(x_imu_flat)  # (B*T, 2*pressure_dim)

        # 分离 gamma 和 beta
        gamma, beta = gamma_beta.chunk(2, dim=-1)  # 各 (B*T, pressure_dim)
        gamma = gamma.view(B, T, D_p)
        beta = beta.view(B, T, D_p)

        # gamma near zero preserves the pressure scale; beta adds a bias.
        # Absolute gain is abs(1+gamma); negative gain reverses the sign.
        # Near-identity modulation also requires beta near zero.
        fused = x_pressure * (1.0 + gamma) + beta

        return fused


class CNNLSTMIMUWithFiLM(nn.Module):
    """Frame-wise pressure CNN and linear IMU embedding, followed by FiLM
    (or concatenation), a two-layer LSTM, temporal attention and regression.
    There is no separate temporal encoder in the IMU branch."""

    def __init__(self, output_dim, pressure_channels=2, imu_feat_dim=19,
                 c1=16, c2=32, pressure_embed=256, imu_embed_dim=64,
                 hidden_size=256, lstm_layers=2, dropout=0.3, use_film=True):
        super().__init__()

        self.use_film = use_film

        # ═══════════════════════════ 压力分支 ═══════════════════════════
        # CNN 编码器（原有的 FrameEncoder）
        self.cnn = FrameEncoder(in_channels=pressure_channels, c1=c1, c2=c2)

        # 获取 CNN 输出维数
        with torch.no_grad():
            dummy = torch.zeros(1, pressure_channels, 66, 30)
            cnn_out_dim = self.cnn(dummy).shape[1]

        # 投影到融合维数
        self.pressure_proj = nn.Linear(cnn_out_dim, pressure_embed)

        # ═══════════════════════════ IMU 分支 ═══════════════════════════
        # Linear embedding of the bilateral IMU feature vector
        self.imu_embed = nn.Linear(imu_feat_dim, imu_embed_dim)

        # ═══════════════════════════ FiLM 融合 ═══════════════════════════
        if use_film:
            self.film = FiLMFusion(imu_embed_dim, pressure_embed, hidden_dim=128, dropout=dropout)
            lstm_input_dim = pressure_embed  # FiLM 输出维数不变
        else:
            # 回退到 concat（用于对照）
            self.film = None
            lstm_input_dim = pressure_embed + imu_embed_dim

        # ═══════════════════════════ 时序处理 ═══════════════════════════
        self.lstm = nn.LSTM(lstm_input_dim, hidden_size, lstm_layers,
                           dropout=dropout if lstm_layers > 1 else 0.0,
                           batch_first=True)

        self.attention = TemporalAttention(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x_pressure, x_imu):
        """
        前向传播

        参数:
        - x_pressure: (B, T, 2, 66, 30) 压力输入
        - x_imu: (B, T, 2, 19) 或 (B, T, 38) IMU 特征

        步骤:
        1. 压力通过 CNN 编码
        2. IMU 通过线性层嵌入
        3. FiLM 融合（或 concat）
        4. LSTM 处理时序
        5. Attention 和全连接输出
        """
        B, T, C, H, W = x_pressure.shape

        # ────── 压力分支 ──────
        x_p = self.cnn(x_pressure.view(B * T, C, H, W))  # (B*T, cnn_out)
        x_p = self.pressure_proj(x_p).view(B, T, -1)      # (B, T, pressure_embed)

        # ────── IMU 分支 ──────
        # 如果 IMU 是 (B, T, 2, 19)，需要先 reshape
        if x_imu.dim() == 4:
            x_imu = x_imu.view(B, T, -1)  # (B, T, 38)

        x_i = self.imu_embed(x_imu)  # (B, T, imu_embed_dim)

        # ────── 融合 ──────
        if self.use_film:
            x_fused = self.film(x_p, x_i)  # (B, T, pressure_embed)
        else:
            # concat（对照组）
            x_fused = torch.cat([x_p, x_i], dim=-1)  # (B, T, pressure_embed + imu_embed_dim)

        # ────── LSTM ──────
        lstm_out, _ = self.lstm(x_fused)  # (B, T, hidden_size)

        # ────── 输出 ──────
        out = self.attention(lstm_out)  # (B, hidden_size)
        out = self.fc(self.dropout(out))  # (B, output_dim)

        return out

In [ ]:
# ── Temporal attention ──────────────────────────────────────
class TemporalAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, lstm_out):
        scores  = self.attn(lstm_out)                # (B, T, 1)
        weights = torch.softmax(scores, dim=1)
        return (weights * lstm_out).sum(dim=1)       # (B, hidden_size)


# ── Frame encoder ───────────────────────────────────────────
class FrameEncoder(nn.Module):
    def __init__(self, in_channels=2, c1=16, c2=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1), nn.ReLU(),
            nn.Conv2d(c1, c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2), nn.ReLU(),
            nn.Conv2d(c2, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

    def forward(self, x):
        return torch.flatten(self.encoder(x), start_dim=1)   # (B, c2)


# ── CNN-LSTM + IMU concat fusion ─────────────────────────────
class CNNLSTMIMURegressor(nn.Module):
    """Legacy concatenation class retained from the BALANCE-inspired branch.
    It is not constructed by make_model(). The active control is selected
    by USE_FILM=False in CNNLSTMIMUWithFiLM."""

    def __init__(self, output_dim, in_channels=2, c1=16, c2=32,
                 pressure_embed=256, imu_feat_dim=12, imu_embed_dim=32,
                 hidden_size=256, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = FrameEncoder(in_channels=in_channels, c1=c1, c2=c2)
        with torch.no_grad():
            encoder_dim = self.encoder(torch.zeros(1, in_channels, 66, 30)).shape[1]

        self.proj = nn.Linear(encoder_dim, pressure_embed)

        # imu_embed_dim = 0 concatenates the supplied IMU features directly,
        # without a separate linear embedding.
        if imu_embed_dim > 0:
            self.imu_proj = nn.Sequential(
                nn.Linear(imu_feat_dim, imu_embed_dim),
                nn.ReLU(),
            )
            imu_width = imu_embed_dim
        else:
            self.imu_proj = nn.Identity()
            imu_width = imu_feat_dim

        fused_dim = pressure_embed + imu_width
        print(f"CNN encoder_dim = {encoder_dim} -> pressure_embed = {pressure_embed}")
        print(f"IMU {imu_feat_dim} -> {imu_width}   |   LSTM input = {fused_dim}")

        self.lstm = nn.LSTM(fused_dim, hidden_size, lstm_layers,
                            dropout=dropout if lstm_layers > 1 else 0.0,
                            batch_first=True)
        self.attention = TemporalAttention(hidden_size)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_size, output_dim)

    def forward(self, x, imu):
        # x   : (B, T, C, H, W)
        # imu : (B, T, 2, 6)
        B, T, C, H, W = x.shape

        x_p = self.encoder(x.view(B * T, C, H, W))
        x_p = self.proj(x_p).view(B, T, -1)          # (B, T, pressure_embed)

        x_i = self.imu_proj(imu.reshape(B, T, -1))   # (B, T, imu_width)

        fused = torch.cat([x_p, x_i], dim=2)         # fusion happens here
        out, _ = self.lstm(fused)
        out = self.attention(out)
        return self.fc(self.dropout(out))


# ── Helpers ────────────────────────────────────────
def make_model():
    """Construct CNNLSTMIMUWithFiLM with the current configuration.
    USE_FILM=True applies affine conditioning; False selects concatenation.
    Both settings regress PRED_FRAMES poses and score the last pose."""
    return CNNLSTMIMUWithFiLM(
        output_dim        = PRED_FRAMES * len(OUTPUT_JOINTS) * 3,
        pressure_channels = PRESSURE_CHANNELS,
        imu_feat_dim      = IMU_FEAT_DIM,
        c1                = CNN_CHANNELS[0],
        c2                = CNN_CHANNELS[1],
        pressure_embed    = PRESSURE_EMBED,
        imu_embed_dim     = IMU_EMBED_DIM,
        hidden_size       = LSTM_HIDDEN_SIZE,
        lstm_layers       = LSTM_LAYERS,
        dropout           = DROPOUT,
        use_film          = USE_FILM,
    ).to(DEVICE)


def make_optimizer(model):
    """Adam parameter groups: CNN 2e-3, LSTM 3e-3, all other modules 1e-3;
    weight decay uses WEIGHT_DECAY. No learning-rate scheduler is used."""
    groups = [
        {"params": model.cnn.parameters(),           "lr": 2e-3},
        {"params": model.pressure_proj.parameters(), "lr": 1e-3},
        {"params": model.imu_embed.parameters(),     "lr": 1e-3},
        {"params": model.lstm.parameters(),          "lr": 3e-3},
        {"params": model.attention.parameters(),     "lr": 1e-3},
        {"params": model.fc.parameters(),            "lr": 1e-3},
    ]
    if model.film is not None:
        groups.append({"params": model.film.parameters(), "lr": 1e-3})
    return torch.optim.Adam(groups, weight_decay=WEIGHT_DECAY)


_m = make_model()
print(f"\nFusion: {'FiLM' if USE_FILM else 'concat'}   "
      f"head: {PRED_FRAMES} frame(s) x {len(OUTPUT_JOINTS)} joints x 3")
print(f"Trainable parameters: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}")
with torch.no_grad():
    _out = _m(torch.zeros(2, SEQ_LEN, PRESSURE_CHANNELS, 66, 30, device=DEVICE),
              torch.zeros(2, SEQ_LEN, N_FEET, IMU_PER_FOOT_DIM, device=DEVICE))
print(f"Forward check: {tuple(_out.shape)} -> "
      f"{(2, PRED_FRAMES, len(OUTPUT_JOINTS), 3)}")
del _m, _out

## 8. Training and evaluation helpers

Optimisation and validation use the composite loss. Position metrics score only
the final predicted pose after inverse normalisation: coordinate-wise MAE in
metres and MSE in square metres. `preds` and `trues` retain these metre-valued
coordinates for the subsequent MPJPE evaluation.

In [ ]:
# Per-joint loss weights, resolved by name so they follow OUTPUT_JOINTS when a
# side is dropped. Hips are nearly constant after normalize_orientation, so they
# are down-weighted; the distal joints carry the error and are boosted.
_JOINT_WEIGHT_BY_PART = {"hip": 0.5, "knee": 1.0, "heel": 2.0, "big-toe-tip": 2.0}
JOINT_WEIGHTS = torch.tensor(
    [_JOINT_WEIGHT_BY_PART[j.split("-", 1)[1]] for j in OUTPUT_JOINTS],
    dtype=torch.float32,
)
N_JOINTS = len(OUTPUT_JOINTS)
print(f"Joint weights ({N_JOINTS}): "
      + ", ".join(f"{j}={w:.1f}" for j, w in zip(OUTPUT_JOINTS, JOINT_WEIGHTS.tolist())))


def weighted_l1(pred, target):
    """Kept for the single-frame path and for comparing against older runs."""
    pred_j    = pred.view(-1, N_JOINTS, 3)
    target_j  = target.view(-1, N_JOINTS, 3)
    per_joint = torch.abs(pred_j - target_j).mean(dim=2)   # (B, J)
    return (per_joint * JOINT_WEIGHTS.to(pred.device)).mean()

criterion = weighted_l1


def _forward(model, batch):
    """Run the model and reshape to (B, K, J, 3)."""
    x   = batch["X"].to(DEVICE)
    imu = batch["IMU"].to(DEVICE)
    y   = batch["Y"].to(DEVICE)                                  # (B, K, J, 3)
    bs  = x.size(0)
    pred = model(x, imu).view(bs, PRED_FRAMES, N_JOINTS, 3)
    return pred, y, bs


def _loss(pred, y, batch, phys):
    if USE_PHYSICS_LOSS:
        stance = batch["STANCE"].to(DEVICE) if "STANCE" in batch else None
        return combined_physical_loss(pred, y, stance, phys, PHYSICS_WEIGHTS)
    return weighted_l1_seq(pred, y), {"kinematic": weighted_l1_seq(pred, y)}


def run_epoch(model, loader, optimizer=None, phys=None, parts_out=None):
    """One pass. `phys` carries the per-fold normalisation and bone references;
    without it the bone term is skipped and the rest still applies."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n_samples = 0.0, 0
    running = {}

    with torch.set_grad_enabled(is_train):
        for batch in loader:
            pred, y, bs = _forward(model, batch)
            loss, parts = _loss(pred, y, batch, phys)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * bs
            n_samples  += bs
            for k, v in parts.items():
                running[k] = running.get(k, 0.0) + float(v) * bs

    if parts_out is not None:
        parts_out.clear()
        parts_out.update({k: v / max(n_samples, 1) for k, v in running.items()})
    return total_loss / max(n_samples, 1)


def evaluate(model, loader, y_mean=None, y_std=None, phys=None):
    """Loss, MSE, MAE, per-joint MAE and raw pred/true arrays in real units.

    Scored on frame -1 only, so MAE stays directly comparable with the
    single-frame runs regardless of PRED_FRAMES.
    """
    model.eval()
    total_loss, n_samples = 0.0, 0
    preds_list, trues_list = [], []

    with torch.no_grad():
        for batch in loader:
            pred, y, bs = _forward(model, batch)
            loss, _ = _loss(pred, y, batch, phys)

            total_loss += loss.item() * bs
            n_samples  += bs

            pred_raw = pred[:, -1].cpu().numpy()                 # (B, J, 3)
            if y_mean is not None and y_std is not None:
                pred_raw = pred_raw * y_std + y_mean
            preds_list.append(pred_raw)
            trues_list.append(batch["Y_raw"].cpu().numpy())

    preds = np.concatenate(preds_list, axis=0)
    trues = np.concatenate(trues_list, axis=0)
    return {
        "loss":          total_loss / max(n_samples, 1),
        "mse":           float(np.mean((preds - trues) ** 2)),
        "mae":           float(np.mean(np.abs(preds - trues))),
        "per_joint_mae": np.mean(np.abs(preds - trues), axis=(0, 2)),
        "preds":         preds,
        "trues":         trues,
    }

print("Helpers defined. Both modalities are passed to the model as model(x, imu).")
print(f"Loss: {'physics-constrained' if USE_PHYSICS_LOSS else 'weighted L1 only'}"
      f"  weights={PHYSICS_WEIGHTS if USE_PHYSICS_LOSS else '-'}")


## 9. Three-participant LOSO cross-validation

For the study dataset, each of three folds holds out one participant. The other
two provide development data. Windows have length 30 and stride 1; a shuffled
15% of their windows is reserved for validation and checkpoint selection.
Adjacent windows overlap, so this internal validation split is not a separate
participant-level generalisation test.

Target mean/std and median bone lengths use the two non-test participants,
including their internal validation frames. The held-out participant is excluded
from these fitted statistics. IMU z-scoring is disabled by default. Model weights
are reinitialised for every fold and the lowest-validation-loss checkpoint is
loaded for evaluation. Seed 42 is retained; deterministic CUDA execution is not
explicitly enforced. The loop uses all loaded IMU-bearing sessions, so retain
only the three study sessions to run the dissertation protocol.

In [ ]:
# Keep each input configuration's checkpoints separate.
MODELS_DIR = MODELS_DIR.parent / f"{MODELS_DIR.name}_{RUN_TAG}"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints  : {MODELS_DIR}")

MODEL_FILE_STEM = "cnn_lstm_imu"

n_subjects     = len(processed_sessions)
fold_results   = []
fold_histories = []

print("=" * 60)
print("CNN-LSTM + IMU — LOSO Cross-Validation")
print(f"Fusion: {'FiLM - IMU modulates the pressure embedding' if USE_FILM else 'concat (control)'}")
print(f"IMU   : {'ZUPT 19-D kinematic block' if USE_ZUPT_FEATURES else 'raw 6 axes'}")
print(f"Loss  : {'weighted L1 + contact/bone/smooth' if USE_PHYSICS_LOSS else 'weighted L1'}")
print("=" * 60)

for test_subj_idx in range(n_subjects):

    test_name   = processed_sessions[test_subj_idx]["name"]
    train_idxs  = [i for i in range(n_subjects) if i != test_subj_idx]
    train_names = [processed_sessions[i]["name"] for i in train_idxs]

    print(f"\n{'─' * 60}")
    print(f"Fold {test_subj_idx + 1}/{n_subjects}  |  "
          f"Test: [{test_name}]  |  Train: {train_names}")
    print(f"{'─' * 60}")

    train_sessions = [copy.deepcopy(processed_sessions[i]) for i in train_idxs]
    test_sessions  = [copy.deepcopy(processed_sessions[test_subj_idx])]

    # Target statistics use non-test participants, including validation frames.
    train_Y = np.concatenate([s["Y"] for s in train_sessions], axis=0)
    Y_mean  = train_Y.mean(axis=0)
    Y_std   = train_Y.std(axis=0) + 1e-6
    for s in train_sessions + test_sessions:
        s["Y"] = (s["Y"] - Y_mean) / Y_std

    # Optional IMU z-scoring on non-test participants; disabled by default.
    if NORMALISE_IMU:
        train_IMU = np.concatenate([s["IMU"] for s in train_sessions], axis=0)
        IMU_mean  = train_IMU.mean(axis=0)
        IMU_std   = train_IMU.std(axis=0) + 1e-6
        for s in train_sessions + test_sessions:
            s["IMU"] = ((s["IMU"] - IMU_mean) / IMU_std).astype(np.float32)
        print(f"  IMU z-scored on {len(train_IMU)} training frames")
    else:
        # Keep the fixed-scaled IMU features without fitted normalisation.
        IMU_mean = IMU_std = None

    # ── Per-fold constants for the physics terms ──────────────
    # train_Y is still in real units here, before the z-score above is applied
    # to the session copies, so the bone references never see the test subject.
    phys_ctx = make_physics_context(Y_mean, Y_std, train_Y) if USE_PHYSICS_LOSS else None
    if phys_ctx is not None:
        print("  Bone refs: " + ", ".join(
            f"{OUTPUT_JOINTS[a].split('-', 1)[1]}→{OUTPUT_JOINTS[b].split('-', 1)[1]}"
            f"={L:.3f}"
            for (a, b), L in zip(BONE_PAIRS_IDX, phys_ctx["bone_ref_list"])))

    # ── Sliding-window items ─────────────────────────────────
    all_train_items = build_items(train_sessions, SEQ_LEN)
    test_items      = build_items(test_sessions,  SEQ_LEN)

    random.seed(SEED)
    random.shuffle(all_train_items)
    n_val            = max(1, int(len(all_train_items) * VAL_RATIO))
    val_items_fold   = all_train_items[-n_val:]
    train_items_fold = all_train_items[:-n_val]

    print(f"  Train sequences : {len(train_items_fold)}")
    print(f"  Val   sequences : {len(val_items_fold)}")
    print(f"  Test  sequences : {len(test_items)}")

    kw = dict(batch_size=BATCH_SIZE, num_workers=0,
              pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(
        SlidingWindowDataset(train_sessions, train_items_fold, SEQ_LEN),
        shuffle=True, **kw)
    val_loader = DataLoader(
        SlidingWindowDataset(train_sessions, val_items_fold, SEQ_LEN),
        shuffle=False, **kw)
    test_loader = DataLoader(
        SlidingWindowDataset(test_sessions, test_items, SEQ_LEN),
        shuffle=False, **kw)

    torch.manual_seed(SEED)
    model     = make_model()
    optimizer = make_optimizer(model)
    model_path = MODELS_DIR / f"{MODEL_FILE_STEM}_fold{test_subj_idx+1}_{test_name}.pt"

    history       = {"train_loss": [], "val_loss": []}
    best_val      = float("inf")
    best_epoch    = -1
    patience_left = PATIENCE

    for epoch in range(1, EPOCHS + 1):
        train_parts = {}
        train_loss = run_epoch(model, train_loader, optimizer=optimizer,
                               phys=phys_ctx, parts_out=train_parts)
        val_loss   = run_epoch(model, val_loader,   optimizer=None, phys=phys_ctx)

        if USE_PHYSICS_LOSS and (epoch == 1 or epoch % 25 == 0):
            print("   parts: " + "  ".join(f"{k}={v:.4f}" for k, v in train_parts.items()))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(f"  Epoch {epoch:03d}/{EPOCHS} | "
              f"train={train_loss:.6f} | val={val_loss:.6f}", end="")

        if val_loss < best_val:
            best_val, best_epoch, patience_left = val_loss, epoch, PATIENCE
            torch.save(model.state_dict(), model_path)
            print("  ✓", end="")
        else:
            patience_left -= 1
        print()

        if patience_left <= 0:
            print(f"  Early stopping at epoch {epoch}.")
            break

    print(f"  Best epoch: {best_epoch},  best val loss: {best_val:.6f}")

    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    results = evaluate(model, test_loader, y_mean=Y_mean, y_std=Y_std, phys=phys_ctx)

    print(f"\n  ── Fold {test_subj_idx+1} Test Results ──")
    print(f"  Test subject : {test_name}")
    print(f"  Test MSE     : {results['mse']:.6f}")
    print(f"  Test MAE     : {results['mae']:.6f}")
    print(f"  Per-joint MAE:")
    for jname, jmae in zip(OUTPUT_JOINTS, results["per_joint_mae"]):
        print(f"    {jname:<22s}: {jmae:.4f}")

    fold_results.append({
        "fold":       test_subj_idx + 1,
        "test_name":  test_name,
        "best_epoch": best_epoch,
        "best_val":   best_val,
        **{k: results[k] for k in ("loss", "mse", "mae", "per_joint_mae", "preds", "trues")},
    })
    fold_histories.append(history)

## 10. Aggregated results across folds

In [ ]:
mses    = [r["mse"] for r in fold_results]
maes    = [r["mae"] for r in fold_results]
pj_maes = np.stack([r["per_joint_mae"] for r in fold_results])  # (3, J)

print("=" * 60)
print("LOSO Cross-Validation — Aggregated Results")
print("=" * 60)

print(f"\n{'Fold':<6} {'Subject':<16} {'MSE':>10} {'MAE':>10}")
print("─" * 46)
for r in fold_results:
    print(f"  {r['fold']:<4} {r['test_name']:<16} "
          f"{r['mse']:>10.6f} {r['mae']:>10.6f}")
print("─" * 46)
print(f"  {'Mean':<20} {np.mean(mses):>10.6f} {np.mean(maes):>10.6f}")
print(f"  {'Std':<20} {np.std(mses):>10.6f} {np.std(maes):>10.6f}")

print("\nPer-joint MAE — mean ± std across folds:")
for j, jname in enumerate(OUTPUT_JOINTS):
    vals = pj_maes[:, j]
    print(f"  {jname:<22s}: {vals.mean():.4f} ± {vals.std():.4f}")

## 10.1 Extended evaluation: position and motion diagnostics

Section A uses inverse-normalised predictions/targets in `fold_results` and
raw targets in `processed_sessions`; it does not reload checkpoints.
It reports overall, distal and per-landmark MPJPE, within-fold error statistics,
a constant-pose comparator, amplitude ratios and Pearson correlations.

The constant pose is the mean over all frames from the two non-test participants.
Its MPJPE and distal MPJPE use the same test windows as the model. Model/baseline
ratios are averaged across folds. Amplitude is `std(pred)/max(std(true),1e-9)`
per landmark and coordinate. Pearson correlation is NaN if either trajectory's
standard deviation is below `1e-12`. Ratios and correlations are dimensionless;
position errors are displayed in millimetres.

Each fold's std/min/max/p95 describes pooled window–landmark Euclidean errors.
Their summary row averages fold statistics; its std is not the standard
deviation of fold MPJPE means. The per-landmark table uses equal-fold means
and population standard deviations (`ddof=0`). Calculations match the
seed-123 extended-evaluation implementation; the training seed remains 42.


In [ ]:
# Section A: position metrics and motion diagnostics, adapted from FiLMseed=123.ipynb.
import numpy as np
import torch
from torch.utils.data import DataLoader

MM      = 1000.0                                     # metres -> millimetres
JN      = OUTPUT_JOINTS
DISTAL  = [i for i, n in enumerate(JN) if not n.endswith("-hip")]
SHORT   = [n.replace("left-", "L-").replace("right-", "R-")
             .replace("-big-toe-tip", "toe") for n in JN]


def _euclid(p, t):
    """(N, J, 3) pairs -> (N, J) per-sample per-joint Euclidean error."""
    return np.linalg.norm(p - t, axis=-1)


def _const_pose(test_name):
    """Training-fold mean pose, the constant predictor evaluate() implies.

    Zero in the network's normalised space inverse-maps to Y_mean, and Y_mean is
    the mean over the training sessions' targets. processed_sessions is never
    mutated by the LOSO loop (it deep-copies), so this reconstructs exactly the
    constant the trained model was competing against.
    """
    tr = [s["Y_raw"] for s in processed_sessions if s["name"] != test_name]
    return np.concatenate(tr, axis=0).mean(axis=0)          # (J, 3)


# ────────────────────────────── Section A ──────────────────────────────────
per_fold = []
for r in fold_results:
    p, t = np.asarray(r["preds"]), np.asarray(r["trues"])   # (N, J, 3), metres
    err  = _euclid(p, t)                                    # (N, J)

    const     = np.broadcast_to(_const_pose(r["test_name"]), t.shape)
    err_const = _euclid(const, t)

    # amplitude: trajectory spread over the test session, per joint per axis
    amp = p.std(axis=0) / np.maximum(t.std(axis=0), 1e-9)   # (J, 3)

    # Pearson r, per joint per axis, over the N windows
    corr = np.zeros((len(JN), 3))
    for j in range(len(JN)):
        for a in range(3):
            pv, tv = p[:, j, a], t[:, j, a]
            if pv.std() < 1e-12 or tv.std() < 1e-12:
                corr[j, a] = np.nan
            else:
                corr[j, a] = np.corrcoef(pv, tv)[0, 1]

    per_fold.append({
        "name":            r["test_name"],
        "err":             err,
        "mpjpe":           err.mean(),
        "mpjpe_distal":    err[:, DISTAL].mean(),
        "per_joint":       err.mean(axis=0),
        "std":             err.std(),
        "min":             err.min(),
        "max":             err.max(),
        "p95":             np.percentile(err, 95),
        "mpjpe_const":     err_const.mean(),
        "const_distal":    err_const[:, DISTAL].mean(),
        "amp":             amp,
        "corr":            corr,
    })

print("=" * 78)
print("EXTENDED EVALUATION -- position errors in mm; ratios/correlation dimensionless")
print("=" * 78)

print(f"\n{'fold':<18}{'MPJPE':>9}{'distal':>9}{'std':>9}"
      f"{'min':>8}{'max':>9}{'p95':>9}")
print("-" * 71)
for f in per_fold:
    print(f"{f['name']:<18}{f['mpjpe']*MM:>9.1f}{f['mpjpe_distal']*MM:>9.1f}"
          f"{f['std']*MM:>9.1f}{f['min']*MM:>8.1f}{f['max']*MM:>9.1f}"
          f"{f['p95']*MM:>9.1f}")
print("-" * 71)
print(f"{'mean':<18}{np.mean([f['mpjpe'] for f in per_fold])*MM:>9.1f}"
      f"{np.mean([f['mpjpe_distal'] for f in per_fold])*MM:>9.1f}"
      f"{np.mean([f['std'] for f in per_fold])*MM:>9.1f}"
      f"{np.mean([f['min'] for f in per_fold])*MM:>8.1f}"
      f"{np.mean([f['max'] for f in per_fold])*MM:>9.1f}"
      f"{np.mean([f['p95'] for f in per_fold])*MM:>9.1f}")
print("  distal = the six joints excluding both hips")
print("  std/min/max/p95 describe each fold; mean row averages fold statistics")

# ── per-joint MPJPE ────────────────────────────────────────────────────────
pj = np.stack([f["per_joint"] for f in per_fold])            # (folds, J)
print(f"\n{'per-joint MPJPE':<22}{'mean':>9}{'std':>9}")
print("-" * 40)
for j, n in enumerate(JN):
    print(f"  {n:<20}{pj[:, j].mean()*MM:>9.1f}{pj[:, j].std()*MM:>9.1f}")

# ── constant-pose baseline and skill ───────────────────────────────────────
print(f"\n{'fold':<18}{'MPJPE':>9}{'const':>9}{'ratio':>8}"
      f"{'MPJPE-d':>10}{'const-d':>10}{'ratio-d':>9}")
print("-" * 73)
for f in per_fold:
    print(f"{f['name']:<18}{f['mpjpe']*MM:>9.1f}{f['mpjpe_const']*MM:>9.1f}"
          f"{f['mpjpe']/f['mpjpe_const']:>8.3f}"
          f"{f['mpjpe_distal']*MM:>10.1f}{f['const_distal']*MM:>10.1f}"
          f"{f['mpjpe_distal']/f['const_distal']:>9.3f}")
print("-" * 73)
_sk  = np.mean([f["mpjpe"] / f["mpjpe_const"] for f in per_fold])
_skd = np.mean([f["mpjpe_distal"] / f["const_distal"] for f in per_fold])
print(f"{'mean ratio':<18}{'':>9}{'':>9}{_sk:>8.3f}{'':>10}{'':>10}{_skd:>9.3f}")
print("\n  const  = predicting the training subjects' mean pose for every frame")
print("  ratio  = MPJPE / const.  < 1.000 the model beats it; >= 1.000 it does not.")
print("           This ratio compares against one constant-pose baseline only.")

# ── amplitude ratio ────────────────────────────────────────────────────────
amp = np.stack([f["amp"] for f in per_fold])                 # (folds, J, 3)
print(f"\n{'amplitude std(pred)/std(true)':<24}{'x':>8}{'y':>8}{'z':>8}{'mean':>9}")
print("-" * 57)
for j, n in enumerate(JN):
    a = np.nanmean(amp[:, j], axis=0)
    print(f"  {n:<22}{a[0]:>8.2f}{a[1]:>8.2f}{a[2]:>8.2f}{a.mean():>9.2f}")
print(f"  {'ALL':<22}{'':>8}{'':>8}{'':>8}{np.nanmean(amp):>9.2f}")
print("\n  1.00 means the prediction swings as far as the target.")
print("  Ratios below 1 indicate lower predicted variation on the corresponding axis.")

# ── Pearson correlation ────────────────────────────────────────────────────
corr = np.stack([f["corr"] for f in per_fold])               # (folds, J, 3)
print(f"\n{'Pearson r':<24}{'x':>8}{'y':>8}{'z':>8}{'mean':>9}")
print("-" * 57)
for j, n in enumerate(JN):
    c = np.nanmean(corr[:, j], axis=0)
    print(f"  {n:<22}{c[0]:>8.2f}{c[1]:>8.2f}{c[2]:>8.2f}{np.nanmean(c):>9.2f}")
print(f"  {'ALL':<22}{'':>8}{'':>8}{'':>8}{np.nanmean(corr):>9.2f}")
print("\n  r is scale-free, so a collapsed-amplitude prediction can still score")
print("  high here. Read it together with the amplitude ratio, never alone.")


## 10.2 Complete per-fold IMU ablation and FiLM diagnostics

Run after Section 10.1 and the LOSO loop, with the matching fold checkpoints in
`MODELS_DIR`. Section B reconstructs each fold's test loader and target
normalisation, loads its checkpoint, and evaluates the same weights with full
IMU input and zero IMU input. It reports overall/distal MPJPE and their changes
in millimetres. No retraining is performed. Missing checkpoints are reported
and skipped; all three are required for a complete three-fold summary.

This reconstruction follows the default `NORMALISE_IMU=False` configuration,
as in seed=123. The FiLM hook remains active during both evaluation passes.
It reports the unweighted mean of batch summaries for `|gamma|`, `std(gamma)`
and `|beta|`, pooling full-input and zero-IMU conditions. These are pooled
implementation diagnostics, not full-input-only or gait-phase statistics.

The evaluation flow and numerical definitions match seed=123. Test-time input
removal measures model sensitivity and can introduce distribution shift.


In [ ]:
# Section B: per-fold IMU-zeroing and FiLM diagnostics, adapted from FiLMseed=123.ipynb.
# Needs the per-fold checkpoints written by the LOSO loop.

def _fold_loader_and_stats(fold_idx, test_name):
    """Rebuild the fold's test loader and its Y_mean / Y_std, as the loop did.

    NORMALISE_IMU is False on the ZUPT path, so the IMU needs no per-fold
    transform and processed_sessions can be used directly.
    """
    tr = [i for i in range(len(processed_sessions))
          if processed_sessions[i]["name"] != test_name]
    train_Y = np.concatenate([processed_sessions[i]["Y_raw"] for i in tr], axis=0)
    y_mean  = train_Y.mean(axis=0)
    y_std   = train_Y.std(axis=0) + 1e-6

    test_sessions = [copy.deepcopy(processed_sessions[fold_idx])]
    test_sessions[0]["Y"] = (test_sessions[0]["Y_raw"] - y_mean) / y_std
    items  = build_items(test_sessions, SEQ_LEN)
    loader = DataLoader(SlidingWindowDataset(test_sessions, items, SEQ_LEN),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                        pin_memory=torch.cuda.is_available())
    return loader, y_mean, y_std


def _run(model, loader, y_mean, y_std, zero_imu=False, film_sink=None):
    """Forward the fold, optionally with the IMU zeroed. Returns (N, J, 3) pairs."""
    model.eval()
    P, T = [], []
    with torch.no_grad():
        for batch in loader:
            x   = batch["X"].to(DEVICE)
            imu = batch["IMU"].to(DEVICE)
            if zero_imu:
                imu = torch.zeros_like(imu)
            out = (model(x, imu)
                   .view(x.size(0), PRED_FRAMES, len(JN), 3)[:, -1]
                   .cpu().numpy())
            P.append(out * y_std + y_mean)
            T.append(batch["Y_raw"].cpu().numpy())
    return np.concatenate(P, axis=0), np.concatenate(T, axis=0)


print("\n" + "=" * 78)
print("IMU ABLATION  --  the same weights, evaluated with the IMU set to zero")
print("=" * 78)

_abl, _film = [], []
for k, r in enumerate(fold_results):
    ckpt = MODELS_DIR / f"{MODEL_FILE_STEM}_fold{r['fold']}_{r['test_name']}.pt"
    if not ckpt.exists():
        print(f"  {r['test_name']}: checkpoint missing ({ckpt.name}) - skipped")
        continue

    loader, y_mean, y_std = _fold_loader_and_stats(k, r["test_name"])
    model = make_model()
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

    # Match seed=123: collect batch summaries during BOTH full-input and
    # zero-IMU passes, then average them without batch-size weighting.
    stats = {}
    handle = None
    if getattr(model, "film", None) is not None:
        def _hook(_m, _inp, out, sink=stats):
            g, b = out.chunk(2, dim=-1)
            sink.setdefault("g_abs", []).append(g.abs().mean().item())
            sink.setdefault("g_std", []).append(g.std().item())
            sink.setdefault("b_abs", []).append(b.abs().mean().item())
        handle = model.film.mlp.register_forward_hook(_hook)

    p_full, t = _run(model, loader, y_mean, y_std, zero_imu=False)
    p_zero, _ = _run(model, loader, y_mean, y_std, zero_imu=True)
    if handle is not None:
        handle.remove()
        _film.append((r["test_name"],
                      float(np.mean(stats["g_abs"])),
                      float(np.mean(stats["g_std"])),
                      float(np.mean(stats["b_abs"]))))

    e_full = _euclid(p_full, t)
    e_zero = _euclid(p_zero, t)
    _abl.append((r["test_name"],
                 e_full.mean(), e_zero.mean(),
                 e_full[:, DISTAL].mean(), e_zero[:, DISTAL].mean()))

if _abl:
    print(f"\n{'fold':<18}{'MPJPE':>10}{'IMU=0':>10}{'delta':>9}"
          f"{'distal':>10}{'IMU=0':>10}{'delta':>9}")
    print("-" * 76)
    for n, a, b, ad, bd in _abl:
        print(f"{n:<18}{a*MM:>10.1f}{b*MM:>10.1f}{(b-a)*MM:>+9.1f}"
              f"{ad*MM:>10.1f}{bd*MM:>10.1f}{(bd-ad)*MM:>+9.1f}")
    print("-" * 76)
    _d  = np.mean([b - a for _, a, b, _, _ in _abl])
    _dd = np.mean([bd - ad for _, _, _, ad, bd in _abl])
    print(f"{'mean delta':<18}{'':>10}{'':>10}{_d*MM:>+9.1f}"
          f"{'':>10}{'':>10}{_dd*MM:>+9.1f}")
    print("\n  Positive delta means IMU zeroing increased error with fixed model weights.")
    print("  Near-zero delta indicates little aggregate change under this intervention.")

if _film:
    print("\n" + "=" * 78)
    print("FiLM MODULATION  --  y = (1 + gamma) * pressure + beta")
    print("=" * 78)
    print(f"\n{'fold':<18}{'|gamma|':>10}{'std(gamma)':>12}{'|beta|':>10}")
    print("-" * 50)
    for n, ga, gs, ba in _film:
        print(f"{n:<18}{ga:>10.4f}{gs:>12.4f}{ba:>10.4f}")
    print("-" * 50)
    print(f"{'mean':<18}{np.mean([x[1] for x in _film]):>10.4f}"
          f"{np.mean([x[2] for x in _film]):>12.4f}"
          f"{np.mean([x[3] for x in _film]):>10.4f}")
    print("\n  |gamma| describes scale adjustment; beta describes the additive adjustment.")
    print("  Near-identity modulation requires both gamma and beta to be near zero.")
    print("  std(gamma) is pooled across batch/frame/channel values in each batch;")
    print("  these diagnostics pool full-input and zero-IMU passes, as in seed=123.")


## 11. Loss curves

One panel per LOSO fold: training vs validation loss, with the early-stopping
best epoch marked. `fold_histories` is filled in by the training loop above.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(fold_histories),
                         figsize=(5 * len(fold_histories), 4), squeeze=False)
axes = axes[0]

for ax, history, r in zip(axes, fold_histories, fold_results):
    ep = range(1, len(history["train_loss"]) + 1)
    ax.plot(ep, history["train_loss"], label="train")
    ax.plot(ep, history["val_loss"],   label="val")
    ax.axvline(r["best_epoch"], color="red", linestyle="--", linewidth=0.8,
               label=f"best ep {r['best_epoch']}")
    ax.set_title(f"Fold {r['fold']} — test: {r['test_name']}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("CNN-LSTM — Training Curves", fontweight="bold")
plt.tight_layout()
plt.savefig("cnn_lstm_loss_curves.png", dpi=150)
plt.show()


## 12. LaTeX summary of position errors

Generate a table from this run's `fold_results`. With the three study sessions,
these are three-fold LOSO results. MSE is in m² and MAE in m, after inverse
normalisation; the summary uses equal-fold means and population standard
deviations. The following plotting helpers use the default bilateral eight-landmark layout.

In [ ]:
print(r"\begin{table}[h]")
print(r"\centering")
print(r"\begin{tabular}{lcc}")
print(r"\toprule")
print(r"Test Subject & MSE (m$^2$) & MAE (m) \\")
print(r"\midrule")
for r in fold_results:
    print(f"  {r['test_name']} (Fold {r['fold']}) & "
          f"{r['mse']:.4f} & {r['mae']:.4f} \\\\")
print(r"\midrule")
print(f"  Mean $\\pm$ Std & "
      f"{np.mean(mses):.4f} $\\pm$ {np.std(mses):.4f} & "
      f"{np.mean(maes):.4f} $\\pm$ {np.std(maes):.4f} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\caption{LOSO cross-validation results (three folds for the study dataset). "
      r"MSE (m$^2$) and MAE (m) are computed after inverse normalisation; "
      r"the summary gives equal-fold means and population standard deviations.}")
print(r"\label{tab:loso_results}")
print(r"\end{table}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D

# ── 参数 ──────────────────────────────────────────────────────
# OUTPUT_JOINTS comes from the joint-setup cell (side-filtered)
CONNECTIONS = [
    (0, 2), (2, 4), (4, 6),   # 左腿: hip→knee→heel→toe
    (1, 3), (3, 5), (5, 7),   # 右腿: hip→knee→heel→toe
    (0, 1),                    # 左髋→右髋
]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. 单帧骨架对比：预测 vs 真实
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def plot_skeleton_comparison(preds, trues, frame_idx=0, fold_name="brahim"):
    """
    preds/trues: (N, 8, 3)
    显示单帧的预测骨架 vs 真实骨架
    """
    fig = plt.figure(figsize=(14, 6))
    titles = ["Ground Truth", "Prediction", "Overlay (GT=blue, Pred=red)"]
    data_pairs = [
        (trues[frame_idx], None),
        (preds[frame_idx], None),
        (trues[frame_idx], preds[frame_idx]),
    ]

    for col, (title, (d1, d2)) in enumerate(zip(titles, data_pairs)):
        ax = fig.add_subplot(1, 3, col+1, projection='3d')
        
        def draw_skeleton(pts, color, label, alpha=1.0):
            ax.scatter(pts[:,0], pts[:,1], pts[:,2],
                       c=color, s=60, zorder=5, label=label, alpha=alpha)
            for i, j in CONNECTIONS:
                ax.plot([pts[i,0], pts[j,0]],
                        [pts[i,1], pts[j,1]],
                        [pts[i,2], pts[j,2]],
                        c=color, lw=2, alpha=alpha)
            # 关节名称标注
            for k, name in enumerate(OUTPUT_JOINTS):
                short = name.replace("left-","L-").replace("right-","R-").replace("-big-toe-tip","Toe")
                ax.text(pts[k,0], pts[k,1], pts[k,2]+0.02, short, fontsize=6, alpha=0.7)
        
        draw_skeleton(d1, '#2196F3', 'GT')
        if d2 is not None:
            draw_skeleton(d2, '#F44336', 'Pred', alpha=0.7)
        
        ax.set_title(f"{title}\n[{fold_name}] frame {frame_idx}", fontsize=10)
        ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
        ax.view_init(elev=15, azim=-70)
        if d2 is not None:
            ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f"skeleton_comparison_fold_{fold_name}.png", dpi=150)
    plt.show()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2. 关节轨迹时序图：预测 vs 真实
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def plot_joint_trajectories(preds, trues, fold_name="brahim", n_frames=300):
    """
    每个关节的X/Y/Z坐标随时间变化，对比预测与真实
    """
    preds = preds[:n_frames]
    trues = trues[:n_frames]
    t = np.arange(len(preds))
    axes_names = ['X', 'Y', 'Z']
    
    fig, axes = plt.subplots(len(OUTPUT_JOINTS), 3, figsize=(15, 20))
    fig.suptitle(f"Joint Trajectories — [{fold_name}]  (first {n_frames} frames)", 
                 fontsize=13, fontweight='bold')
    
    for j, jname in enumerate(OUTPUT_JOINTS):
        for ax_idx, ax_name in enumerate(axes_names):
            ax = axes[j, ax_idx]
            ax.plot(t, trues[:, j, ax_idx], 'b-', lw=1.5, label='GT', alpha=0.8)
            ax.plot(t, preds[:, j, ax_idx], 'r--', lw=1.2, label='Pred', alpha=0.8)
            err = np.abs(preds[:, j, ax_idx] - trues[:, j, ax_idx])
            ax.fill_between(t, trues[:, j, ax_idx] - err, 
                            trues[:, j, ax_idx] + err, 
                            alpha=0.15, color='red', label='|error|')
            ax.set_ylabel(ax_name, fontsize=8)
            if j == 0:
                ax.set_title(ax_name, fontsize=9)
            if ax_idx == 0:
                short = jname.replace("left-","L-").replace("right-","R-")
                ax.set_ylabel(short + f"\n{ax_name}", fontsize=7)
            if j == 0 and ax_idx == 0:
                ax.legend(fontsize=6, loc='upper right')
            ax.tick_params(labelsize=6)
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"trajectories_fold_{fold_name}.png", dpi=150)
    plt.show()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3. 关节误差热力图（所有帧）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def plot_error_heatmap(preds, trues, fold_name="brahim"):
    """
    热力图：横轴=时间帧，纵轴=关节，颜色=MAE
    直观看到哪些时刻、哪些关节误差最大
    """
    # (N, 8) — 每帧每关节的MAE
    per_frame_mae = np.mean(np.abs(preds - trues), axis=2)
    
    fig, ax = plt.subplots(figsize=(16, 4))
    im = ax.imshow(per_frame_mae.T, aspect='auto', 
                   cmap='RdYlGn_r', interpolation='nearest')
    ax.set_yticks(range(len(OUTPUT_JOINTS)))
    ax.set_yticklabels([j.replace("left-","L-").replace("right-","R-")
                        .replace("-big-toe-tip","Toe") 
                        for j in OUTPUT_JOINTS], fontsize=9)
    ax.set_xlabel("Frame index", fontsize=10)
    ax.set_title(f"Per-frame Per-joint MAE — [{fold_name}]  (green=low, red=high)", 
                 fontsize=11)
    plt.colorbar(im, ax=ax, label='MAE')
    plt.tight_layout()
    plt.savefig(f"error_heatmap_fold_{fold_name}.png", dpi=150)
    plt.show()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4. 步态动画：预测 vs 真实（GIF / mp4）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def animate_gait(preds, trues, fold_name="brahim", 
                 n_frames=120, interval=80, save_gif=True):
    """
    动画：左图真实骨架，右图预测骨架，实时播放步态
    """
    preds = preds[:n_frames]
    trues = trues[:n_frames]
    
    # 统一坐标范围
    all_pts = np.concatenate([preds, trues], axis=0)
    lims = [(all_pts[:,:,i].min()-0.1, all_pts[:,:,i].max()+0.1) 
            for i in range(3)]
    
    fig = plt.figure(figsize=(12, 5))
    ax1 = fig.add_subplot(121, projection='3d')
    ax2 = fig.add_subplot(122, projection='3d')
    
    def setup_ax(ax, title):
        ax.set_xlim(*lims[0]); ax.set_ylim(*lims[1]); ax.set_zlim(*lims[2])
        ax.set_xlabel('X', fontsize=7); ax.set_ylabel('Y', fontsize=7)
        ax.set_zlabel('Z', fontsize=7)
        ax.set_title(title, fontsize=10)
        ax.view_init(elev=15, azim=-70)
        ax.tick_params(labelsize=6)
    
    setup_ax(ax1, f"Ground Truth [{fold_name}]")
    setup_ax(ax2, f"Prediction [{fold_name}]")
    fig.suptitle("Gait Animation", fontsize=12, fontweight='bold')
    
    scat1 = ax1.scatter([], [], [], c='#2196F3', s=50)
    scat2 = ax2.scatter([], [], [], c='#F44336', s=50)
    lines1 = [ax1.plot([], [], [], 'b-', lw=2)[0] for _ in CONNECTIONS]
    lines2 = [ax2.plot([], [], [], 'r-', lw=2)[0] for _ in CONNECTIONS]
    frame_txt = fig.text(0.5, 0.02, '', ha='center', fontsize=9)
    
    def update(frame):
        for pts, scat, lines in [(trues[frame], scat1, lines1),
                                  (preds[frame], scat2, lines2)]:
            scat._offsets3d = (pts[:,0], pts[:,1], pts[:,2])
            for (i, j), line in zip(CONNECTIONS, lines):
                line.set_data([pts[i,0], pts[j,0]], [pts[i,1], pts[j,1]])
                line.set_3d_properties([pts[i,2], pts[j,2]])
        frame_txt.set_text(f"Frame {frame}/{n_frames}")
        return [scat1, scat2] + lines1 + lines2 + [frame_txt]
    
    ani = animation.FuncAnimation(fig, update, frames=n_frames,
                                  interval=interval, blit=True)
    if save_gif:
        ani.save(f"gait_{fold_name}.gif", writer='pillow', fps=1000//interval)
        print(f"Saved: gait_{fold_name}.gif")
    plt.show()
    return ani


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 使用方式：在fold_results拿到数据后直接调用
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
for result in fold_results:
    name   = result["test_name"]
    preds  = result["preds"]   # (N, 8, 3)
    trues  = result["trues"]   # (N, 8, 3)
    
    print(f"\n=== Visualizing [{name}] ===")
    plot_skeleton_comparison(preds, trues, frame_idx=50, fold_name=name)
    plot_joint_trajectories(preds, trues, fold_name=name, n_frames=300)
    plot_error_heatmap(preds, trues, fold_name=name)
    animate_gait(preds, trues, fold_name=name, n_frames=120, save_gif=True)



## 13. Recorded predictions: pose, pressure and per-landmark error

Render stored test predictions as frame images and a GIF. This is an offline
visualisation, not a live sensor-inference loop. Panel layout is adapted from
BALANCE's `APP_Fall_Risk_Prediction/utils/make_live_demo.py`.

The panels show reference/predicted 3D landmarks, pressure for each selected
foot, and per-landmark coordinate MAE in metres. IMU is used by the model but
is not displayed as a separate panel. Pressure maps receive additional
masking, temporal smoothing, upscaling and blur for display only.
Set `DEMO_N_FRAMES=None` to render all evaluated frames.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Offline prediction visualisation — panel layout adapted from BALANCE
#   APP_Fall_Risk_Prediction/utils/make_live_demo.py
#   (LiveDemo._generate_single_frame + LiveDemo.create_gif)
#
# BALANCE panel                  ->  here
#   3D pose, GT vs Pred          ->  3D skeleton, GT vs Pred (with bones)
#   left / right foot pressure   ->  left / right foot pressure (model input)
#   IMU accelerometer bars       ->  omitted from display; IMU remains a model input
#   fall-probability bars        ->  per-joint MAE bars for the current frame
#
# Same rendering recipe as BALANCE: one PNG per frame, then imageio stitches
# them into a GIF. Pressure goes through mask -> pixel EMA -> integer upscale
# -> mask-normalised Gaussian blur, so the heat map reads as a smooth footprint
# instead of a block mosaic.
# ─────────────────────────────────────────────────────────────────────────
import inspect
from pathlib import Path

import imageio.v2 as imageio
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable
from tqdm import tqdm

DEMO_MODEL_TAG = f"cnn_lstm_imu_{RUN_TAG}"
DEMO_ROOT      = Path("notebook")
DEMO_N_FRAMES  = 150     # frames per fold; None = whole test set (slow)
DEMO_START     = 0
DEMO_FPS       = 10

PRESSURE_CFG = {
    "vmin": 0.0,        # insoles are normalised to [0, 1], 1 = fully pressed
    "vmax": 1.0,
    "upscale": 4,       # on top of the 2x already applied in preprocessing
    "blur_sigma": 1.0,
    "ema_alpha": 0.3,   # 0 disables temporal smoothing
    "cmap": "viridis",
}

# Same array as part2_preprocessing.ipynb. Needed here because preprocess_sessions()
# replaced the out-of-foot NaNs with 0, so the footprint outline is no longer in X.
LEFT_FOOT_MASK_RAW = np.array([
    [0,0,0,0,0,0,1,1,1,1,0,0,0,0,0],
    [0,0,0,0,0,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,0,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,1,1,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,1,1,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,1,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,1,1,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,0,0,0,0,0],
    [0,0,0,1,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,1,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,0,1,1,1,1,1,1,0,0,0,0],
    [0,0,0,0,0,1,1,1,1,1,0,0,0,0,0],
], dtype=bool)
RIGHT_FOOT_MASK_RAW = np.fliplr(LEFT_FOOT_MASK_RAW)

# Bones, resolved by name so this works whatever order OUTPUT_JOINTS is in
# (the dual-head notebooks regroup it).
_JIDX = {name: i for i, name in enumerate(OUTPUT_JOINTS)}
DEMO_BONES = [(_JIDX[a], _JIDX[b]) for a, b in [
    ("left-hip",  "left-knee"),  ("left-knee",  "left-heel"),  ("left-heel",  "left-big-toe-tip"),
    ("right-hip", "right-knee"), ("right-knee", "right-heel"), ("right-heel", "right-big-toe-tip"),
    ("left-hip",  "right-hip"),
] if a in _JIDX and b in _JIDX]   # drops bones whose side was not regressed
DEMO_SHORT = [j.replace("left-", "L-").replace("right-", "R-").replace("-big-toe-tip", "Toe")
              for j in OUTPUT_JOINTS]

_demo_cmap = plt.get_cmap(PRESSURE_CFG["cmap"]).copy()
_demo_cmap.set_bad(alpha=0.0)          # outside the foot outline -> transparent


def _foot_masks(h, w):
    """Raw 33x15 masks blown up to the stored insole resolution (66x30 here)."""
    up_h = h // LEFT_FOOT_MASK_RAW.shape[0]
    up_w = w // LEFT_FOOT_MASK_RAW.shape[1]
    grow = lambda m: np.repeat(np.repeat(m, up_h, axis=0), up_w, axis=1)
    return grow(LEFT_FOOT_MASK_RAW), grow(RIGHT_FOOT_MASK_RAW)


def _process_pressure(raw, mask, ema_prev, cfg):
    """mask -> pixel EMA -> integer upscale -> mask-normalised Gaussian blur.

    Returns the display array (NaN outside the foot) and the EMA state to carry
    into the next frame.
    """
    data = np.where(mask, np.asarray(raw, dtype=float), np.nan)

    alpha = float(cfg.get("ema_alpha", 0.0))
    if alpha > 0.0:
        prev = data if ema_prev is None else ema_prev
        blended = alpha * np.nan_to_num(data) + (1.0 - alpha) * np.nan_to_num(prev)
        data = np.where(mask, blended, np.nan)
    ema_state = data.copy()

    up = int(cfg.get("upscale", 1))
    if up > 1:
        data = np.repeat(np.repeat(data, up, axis=0), up, axis=1)
        mask = np.repeat(np.repeat(mask, up, axis=0), up, axis=1)
        data = np.where(mask, data, np.nan)

    sigma = float(cfg.get("blur_sigma", 0.0)) * max(1.0, up * 0.4)
    if sigma > 0 and not np.all(np.isnan(data)):
        radius = int(max(1, round(3 * sigma)))
        xs = np.arange(-radius, radius + 1)
        kern = np.exp(-0.5 * (xs / sigma) ** 2)
        kern /= kern.sum()

        def sep_conv(arr):
            arr = np.apply_along_axis(lambda m: np.convolve(m, kern, mode="same"), 0, arr)
            return np.apply_along_axis(lambda m: np.convolve(m, kern, mode="same"), 1, arr)

        valid = ~np.isnan(data)
        num = sep_conv(np.where(valid, data, 0.0))
        den = sep_conv(valid.astype(float))
        data = np.where(valid, num / np.maximum(den, 1e-6), np.nan)

    return data, ema_state


def collect_fold_pressure(fold_idx, n_rows):
    """Recover pressure at evaluated window endpoints with build_items().
    X contains the preprocessed maps for INPUT_FEET; no velocity channels
    are added in this notebook. Rendering applies additional display smoothing.
    """
    session = processed_sessions[fold_idx]
    kwargs = {}
    if "stride" in inspect.signature(build_items).parameters:
        kwargs["stride"] = globals().get("SEQ_STRIDE", 1)
    items = build_items([session], SEQ_LEN, **kwargs)

    end_idx = np.array([end for _, end in items], dtype=int)
    if len(end_idx) < n_rows:
        raise ValueError(f"fold {fold_idx}: build_items gave {len(end_idx)} windows "
                         f"but there are {n_rows} predictions - did the fold order change?")
    return session["X"][end_idx[:n_rows]]   # already sliced to INPUT_FEET


def make_live_demo(preds, trues, pressure, fold_name,
                   model_tag=None, n_frames=None, start=0, fps=None, out_root=None):
    """Render one PNG per frame, then stitch them into a GIF.

    preds/trues : (N, J, 3) in raw joint units
    pressure    : (N, 2, H, W) model input, channel 0 = left foot
    """
    model_tag = model_tag or DEMO_MODEL_TAG
    fps       = fps or DEMO_FPS
    out_root  = Path(out_root) if out_root else DEMO_ROOT

    total = min(len(preds), len(pressure))
    stop  = total if n_frames is None else min(total, start + n_frames)
    frames = range(start, stop)

    out_dir    = out_root / f"live_demo_{model_tag}_{fold_name}"
    frames_dir = out_dir / "frames"
    frames_dir.mkdir(parents=True, exist_ok=True)
    # Just "demo.gif": the folder already carries the tag, and repeating it in
    # the filename pushed the full path past Windows' 260-character limit once
    # the model tag grew (cnn_transformer_imu_inLR_outLR).
    gif_path = out_dir / "demo.gif"

    # Fixed axis limits over the rendered span so the skeleton does not jitter.
    span = np.concatenate([preds[start:stop], trues[start:stop]], axis=0)
    lim = [(span[:, :, i].min() - 0.05, span[:, :, i].max() + 0.05) for i in range(3)]

    per_joint_mae = np.abs(preds[start:stop] - trues[start:stop]).mean(axis=2)   # (n, J)
    mae_top = float(per_joint_mae.max()) * 1.3 + 1e-6
    fold_mean = per_joint_mae.mean(axis=0)

    # Keep the body proportioned instead of stretched to fill a wide panel.
    box_aspect = (lim[0][1] - lim[0][0], lim[2][1] - lim[2][0], lim[1][1] - lim[1][0])

    left_mask, right_mask = _foot_masks(pressure.shape[2], pressure.shape[3])
    ema_state = [None] * len(INPUT_FEET)
    bar_colors = ["#90A4AE"] * 4 + ["#F44336"] * 4      # highlight the distal joints

    paths = []
    for frame_idx in tqdm(frames, desc=f"[{fold_name}] frames"):
        fig = plt.figure(figsize=(12, 11), dpi=110)
        gs = gridspec.GridSpec(3, 2, height_ratios=[2.8, 2.4, 1.1],
                               hspace=0.34, wspace=0.28)
        ax_3d = fig.add_subplot(gs[0, :], projection="3d")
        # One pressure panel per selected insole; a single foot spans the row.
        if len(INPUT_FEET) == 1:
            ax_feet = [fig.add_subplot(gs[1, :])]
        else:
            ax_feet = [fig.add_subplot(gs[1, k]) for k in range(len(INPUT_FEET))]
        ax_e  = fig.add_subplot(gs[2, :])

        fig.suptitle(f"{model_tag.upper()} | test subject [{fold_name}] | frame {frame_idx}"
                     f"  —  pelvis-relative 3D pose from plantar pressure",
                     fontsize=14, fontweight="bold")

        # ── 3D skeleton: GT vs prediction ────────────────────────────────
        # Plot (X, Z, Y): Y is the vertical axis and grows downward after
        # normalize_orientation, so the z-axis is inverted to stand the body up.
        for pts, color, label, marker in ((trues[frame_idx], "#2196F3", "Ground truth", "o"),
                                          (preds[frame_idx], "#F44336", "Prediction", "^")):
            ax_3d.scatter(pts[:, 0], pts[:, 2], pts[:, 1],
                          c=color, marker=marker, s=45, alpha=0.9, label=label)
            for i, j in DEMO_BONES:
                ax_3d.plot([pts[i, 0], pts[j, 0]], [pts[i, 2], pts[j, 2]],
                           [pts[i, 1], pts[j, 1]], c=color, lw=2, alpha=0.8)
        ax_3d.scatter([0], [0], [0], c="black", marker="*", s=110,
                      label="Mid-hip (origin)")

        ax_3d.set_xlim(*lim[0]); ax_3d.set_ylim(*lim[2]); ax_3d.set_zlim(*lim[1])
        ax_3d.set_box_aspect(box_aspect)
        ax_3d.invert_zaxis()
        ax_3d.set_xlabel("X  fore-aft", labelpad=8)
        ax_3d.set_ylabel("Z  lateral", labelpad=10)
        ax_3d.set_zlabel("Y  vertical", labelpad=6)
        ax_3d.tick_params(labelsize=8)
        ax_3d.set_title("Pose estimation (pelvis-relative)", fontsize=11)
        ax_3d.view_init(elev=18, azim=-72)
        ax_3d.grid(True, alpha=0.3)
        ax_3d.legend(fontsize=9, loc="upper right")

        # ── Model input: left / right plantar pressure ───────────────────
        for side, (ax, foot_name) in enumerate(zip(ax_feet, INPUT_FEET)):
            mask  = left_mask if foot_name == "left" else right_mask
            title = f"Input: {foot_name} foot pressure"
            disp, ema_state[side] = _process_pressure(
                pressure[frame_idx, side], mask, ema_state[side], PRESSURE_CFG)
            im = ax.imshow(disp, cmap=_demo_cmap,
                           vmin=PRESSURE_CFG["vmin"], vmax=PRESSURE_CFG["vmax"],
                           interpolation="nearest", extent=[0, 15, 33, 0])
            ax.set_title(title, fontsize=11)
            ax.set_xlabel("Width (15 sensors)"); ax.set_ylabel("Height (33 sensors)")
            ax.set_xticks(range(0, 16, 3)); ax.set_yticks(range(0, 34, 6))
            cax = make_axes_locatable(ax).append_axes("right", size="4%", pad=0.05)
            plt.colorbar(im, cax=cax, label="Normalised pressure")

        # ── Per-joint error for this frame ───────────────────────────────
        x = np.arange(len(OUTPUT_JOINTS))
        bars = ax_e.bar(x, per_joint_mae[frame_idx - start], color=bar_colors, alpha=0.85)
        ax_e.plot(x, fold_mean, "k_", markersize=22, markeredgewidth=2,
                  linestyle="none", label="fold mean")
        for bar, value in zip(bars, per_joint_mae[frame_idx - start]):
            ax_e.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + mae_top * 0.04,
                      f"{value:.3f}", ha="center", va="bottom", fontsize=8)
        ax_e.set_xticks(x); ax_e.set_xticklabels(DEMO_SHORT, fontsize=9)
        ax_e.set_ylim(0, mae_top)
        ax_e.set_ylabel("MAE")
        ax_e.set_title("Per-joint error this frame (red = distal joints)", fontsize=11)
        ax_e.grid(True, axis="y", alpha=0.3)
        ax_e.legend(fontsize=9, loc="upper right")

        path = frames_dir / f"frame_{frame_idx:04d}.png"
        fig.savefig(path, bbox_inches="tight", dpi=110)
        plt.close(fig)
        paths.append(path)

    with imageio.get_writer(gif_path, mode="I", duration=1.0 / fps, loop=0) as writer:
        for path in paths:
            writer.append_data(imageio.imread(path))

    print(f"[{fold_name}] {len(paths)} frames -> {frames_dir}")
    print(f"[{fold_name}] GIF -> {gif_path}")
    return gif_path


# ── Run for every fold ───────────────────────────────────────────────────
for result in fold_results:
    fold_idx = result["fold"] - 1          # fold k tests processed_sessions[k-1]
    preds, trues = result["preds"], result["trues"]
    pressure = collect_fold_pressure(fold_idx, len(preds))
    make_live_demo(preds, trues, pressure, result["test_name"],
                   n_frames=DEMO_N_FRAMES, start=DEMO_START)

## 14. Supplementary geometric angle diagnostics

Angles are derived from predicted and reference landmarks; they complement the
position metrics and are not clinical joint-angle measurements.

| Label | Geometric definition |
|---|---|
| `knee` | Angle at the knee between the knee-to-hip and knee-to-heel vectors |
| `ankle*` | Angle at the heel between the heel-to-knee and heel-to-toe vectors |
| `thigh` | Hip-to-knee inclination from vertical |

The target convention is X fore-aft, Y downward vertical and Z medio-lateral.
Unsigned angles use three-dimensional `arccos`. The supplementary signed
quantity uses `atan2((v1 × v2) · normal, v1 · v2)` with normal +Z. Because the
vectors are not projected first, this is not a strict XY-plane angle when they
have nonzero lateral components. The eight-landmark layout has no ankle marker;
`ankle*` and `thigh` are geometric proxies rather than anatomical ankle/hip angles.

For each held-out participant, the constant comparator uses the mean reference
angle from the other participants' evaluated frames. A model/constant MAE ratio
below 1 indicates lower error than that comparator; a ratio at or above 1 does
not by itself establish that the model learned no transferable information.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Joint-angle error, in degrees
#
# Supplementary geometric angles from eight landmarks.
# X is fore-aft, Y is downward vertical, Z is medio-lateral.
# The knee-heel-toe angle is an ankle proxy, not an anatomical ankle angle.
# The signed quantity below uses a +Z cross component and a full 3D dot
# product; without projection it is not a strict sagittal-plane angle.

import numpy as np
import matplotlib.pyplot as plt

VERTICAL_DOWN = np.array([0.0, 1.0, 0.0])    # +Y points at the floor
SAGITTAL_NORMAL = np.array([0.0, 0.0, 1.0])  # XY is the sagittal plane

ANGLE_N_FRAMES = 300      # how many frames the trajectory plot shows


def _unit(v):
    return v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-8)


def joint_angle(p_prox, p_joint, p_dist):
    """Unsigned angle at p_joint, degrees. Arrays are (T, 3)."""
    v1, v2 = p_prox - p_joint, p_dist - p_joint
    cos = (v1 * v2).sum(-1) / (np.linalg.norm(v1, axis=-1) * np.linalg.norm(v2, axis=-1) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))


def signed_sagittal_angle(v1, v2, normal=SAGITTAL_NORMAL):
    """Signed atan2 quantity from the normal component of cross and full 3D
    dot products. It equals a planar angle only for vectors in that plane."""
    sin = np.cross(v1, v2) @ normal
    cos = (v1 * v2).sum(-1)
    return np.degrees(np.arctan2(sin, cos))


def build_angle_defs(joint_names):
    """Angles computable from whichever joints this run actually regresses."""
    idx = {n: i for i, n in enumerate(joint_names)}
    defs = []
    for side, tag in (("left", "L"), ("right", "R")):
        hip, knee = f"{side}-hip", f"{side}-knee"
        heel, toe = f"{side}-heel", f"{side}-big-toe-tip"
        if all(j in idx for j in (hip, knee, heel)):
            defs.append((f"{tag} knee", "triplet", (idx[hip], idx[knee], idx[heel])))
        if all(j in idx for j in (knee, heel, toe)):
            defs.append((f"{tag} ankle*", "triplet", (idx[knee], idx[heel], idx[toe])))
        if all(j in idx for j in (hip, knee)):
            defs.append((f"{tag} thigh", "vertical", (idx[hip], idx[knee])))
    return defs


def compute_angles(coords, defs):
    """coords (T, J, 3) -> {name: (unsigned_deg, signed_deg)}."""
    out = {}
    for name, kind, ids in defs:
        if kind == "triplet":
            a, b, c = ids
            uns = joint_angle(coords[:, a], coords[:, b], coords[:, c])
            sgn = signed_sagittal_angle(coords[:, a] - coords[:, b], coords[:, c] - coords[:, b])
        else:                                   # thigh inclination from vertical
            h, k = ids
            down = np.broadcast_to(VERTICAL_DOWN, coords[:, h].shape)
            thigh = coords[:, k] - coords[:, h]
            uns = joint_angle(coords[:, h] + down, coords[:, h], coords[:, k])
            sgn = signed_sagittal_angle(down, thigh)
        out[name] = (uns, sgn)
    return out


ANGLE_DEFS = build_angle_defs(OUTPUT_JOINTS)
ANGLE_NAMES = [d[0] for d in ANGLE_DEFS]
print(f"Angles computed from {len(OUTPUT_JOINTS)} joints: {ANGLE_NAMES}")

# ── Per-fold angles ──────────────────────────────────────────────────────
fold_angles = []
for r in fold_results:
    fold_angles.append({
        "test_name": r["test_name"],
        "pred": compute_angles(r["preds"], ANGLE_DEFS),
        "true": compute_angles(r["trues"], ANGLE_DEFS),
    })

# Constant angle comparator: use reference angles from the evaluated
# windows of the other participants. It is separate from position metrics.
baseline = {}
for i, fa in enumerate(fold_angles):
    others = [fold_angles[j] for j in range(len(fold_angles)) if j != i]
    baseline[fa["test_name"]] = {
        name: (np.mean(np.concatenate([o["true"][name][0] for o in others])),
               np.mean(np.concatenate([o["true"][name][1] for o in others])))
        for name in ANGLE_NAMES
    } if others else {name: (0.0, 0.0) for name in ANGLE_NAMES}


def _mae_rmse(err):
    return float(np.mean(np.abs(err))), float(np.sqrt(np.mean(err ** 2)))


rows = {name: {"mae": [], "rmse": [], "s_mae": [], "s_rmse": [],
               "b_mae": [], "b_rmse": []} for name in ANGLE_NAMES}
for fa in fold_angles:
    for name in ANGLE_NAMES:
        pu, ps = fa["pred"][name]
        tu, ts = fa["true"][name]
        m, r_ = _mae_rmse(pu - tu)
        sm, sr = _mae_rmse((ps - ts + 180) % 360 - 180)      # wrap to +/-180
        bu, _bs = baseline[fa["test_name"]][name]
        bm, br = _mae_rmse(bu - tu)
        rows[name]["mae"].append(m);    rows[name]["rmse"].append(r_)
        rows[name]["s_mae"].append(sm); rows[name]["s_rmse"].append(sr)
        rows[name]["b_mae"].append(bm); rows[name]["b_rmse"].append(br)

print("\n" + "=" * 88)
print("JOINT ANGLE ERROR (degrees) - mean +/- std across LOSO folds")
print("=" * 88)
print(f"{'angle':<12}{'MAE':>16}{'RMSE':>16}{'signed MAE':>16}"
      f"{'constant MAE':>16}{'vs const':>10}")
for name in ANGLE_NAMES:
    d = rows[name]
    mae, rmse = np.array(d["mae"]), np.array(d["rmse"])
    smae, bmae = np.array(d["s_mae"]), np.array(d["b_mae"])
    ratio = mae.mean() / (bmae.mean() + 1e-9)
    print(f"{name:<12}{f'{mae.mean():.2f} ± {mae.std():.2f}':>16}"
          f"{f'{rmse.mean():.2f} ± {rmse.std():.2f}':>16}"
          f"{f'{smae.mean():.2f} ± {smae.std():.2f}':>16}"
          f"{f'{bmae.mean():.2f}':>16}{ratio:>9.2f}x")

print("\n  MAE / RMSE   unsigned three-point angle (arccos), the usual literature form")
print("  signed MAE   atan2 diagnostic using +Z cross and full 3D dot (no projection)")
print("  constant     mean reference angle from other participants' evaluated frames")
print("  vs const     < 1.00 means the model beats that constant; >= 1.00 means it does not")
print("\n  ankle* is the knee-heel-toe angle - there is no ankle marker in this layout.")
print("  thigh is inclination from vertical, not an anatomical hip angle.")

print(f"\n{'angle':<12}" + "".join(f"{fa['test_name'][:14]:>16}" for fa in fold_angles))
for name in ANGLE_NAMES:
    print(f"{name:<12}" + "".join(f"{v:>16.2f}" for v in rows[name]["mae"]))
print("  (per-fold unsigned MAE, degrees)")

# ── Plots ────────────────────────────────────────────────────────────────
n_ang = len(ANGLE_NAMES)
for fa in fold_angles:
    fig, axes = plt.subplots(n_ang, 2, figsize=(15, 2.3 * n_ang), squeeze=False,
                             gridspec_kw={"width_ratios": [3, 1]})
    fig.suptitle(f"Joint angles — [{fa['test_name']}]", fontsize=13, fontweight="bold")
    for i, name in enumerate(ANGLE_NAMES):
        pu, _ = fa["pred"][name]
        tu, _ = fa["true"][name]
        n = min(ANGLE_N_FRAMES, len(pu))
        t = np.arange(n)

        ax = axes[i, 0]
        ax.plot(t, tu[:n], "b-", lw=1.5, label="GT")
        ax.plot(t, pu[:n], "r--", lw=1.2, label="Pred")
        ax.set_ylabel(f"{name}\n(deg)", fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=7)
        if i == 0:
            ax.legend(fontsize=7, loc="upper right")
        if i == n_ang - 1:
            ax.set_xlabel("Frame")

        err = pu - tu
        mae, rmse = _mae_rmse(err)
        ax = axes[i, 1]
        ax.hist(err, bins=40, color="#F44336", alpha=0.75)
        ax.axvline(0, color="k", lw=0.8)
        ax.set_title(f"MAE {mae:.1f}°   RMSE {rmse:.1f}°", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)
        if i == n_ang - 1:
            ax.set_xlabel("Pred − GT (deg)")

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(f"angle_trajectories_{fa['test_name']}.png", dpi=150)
    plt.show()

# Summary bar chart: model against the constant predictor
fig, ax = plt.subplots(figsize=(1.6 * n_ang + 3, 4))
x = np.arange(n_ang)
w = 0.26
ax.bar(x - w, [np.mean(rows[n]["mae"]) for n in ANGLE_NAMES], w,
       yerr=[np.std(rows[n]["mae"]) for n in ANGLE_NAMES], capsize=3,
       label="model MAE", color="steelblue")
ax.bar(x, [np.mean(rows[n]["rmse"]) for n in ANGLE_NAMES], w,
       yerr=[np.std(rows[n]["rmse"]) for n in ANGLE_NAMES], capsize=3,
       label="model RMSE", color="#7E57C2")
ax.bar(x + w, [np.mean(rows[n]["b_mae"]) for n in ANGLE_NAMES], w,
       label="constant MAE", color="#BDBDBD")
ax.set_xticks(x); ax.set_xticklabels(ANGLE_NAMES, rotation=20, ha="right")
ax.set_ylabel("degrees")
ax.set_title("Joint angle error vs a constant predictor (mean ± std over folds)")
ax.legend(fontsize=9)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("angle_error_summary.png", dpi=150)
plt.show()

## 15. Optional single-fold zero-IMU MAE helper

The complete three-fold MPJPE ablation and FiLM diagnostics run in Section 10.2.
The helper below is retained for inspecting per-landmark coordinate MAE for one
supplied model/loader pair. Pass that fold's matching target mean/std; it does
not load checkpoints or repeat the complete evaluation automatically.


In [ ]:
# Optional single-fold MAE helper; full per-fold MPJPE ablation is in 10.2.

def evaluate_imu_ablated(model, loader, y_mean, y_std):
    model.eval()
    preds_list, trues_list = [], []
    with torch.no_grad():
        for batch in loader:
            x     = batch["X"].to(DEVICE)
            imu   = torch.zeros_like(batch["IMU"]).to(DEVICE)   # <- ablated
            y_raw = batch["Y_raw"].cpu().numpy()
            pred  = (model(x, imu)
                     .view(x.size(0), PRED_FRAMES, len(OUTPUT_JOINTS), 3)[:, -1]
                     .cpu().numpy())          # score frame -1, as evaluate() does
            preds_list.append(pred * y_std + y_mean)
            trues_list.append(y_raw)
    preds = np.concatenate(preds_list, axis=0)
    trues = np.concatenate(trues_list, axis=0)
    return np.mean(np.abs(preds - trues), axis=(0, 2))


print("Complete per-fold IMU ablation and FiLM diagnostics: run Section 10.2.")
